# 아기캥거루 Kaggle 제출 파이프라인 V2.1 — V6.0.5 CURRENT FIXED

이 노트북은 **재학습하지 않고**, 현재 V6.0.5 공통 파이프라인에서 완료된
YOLO11s / YOLO11m / YOLO12m의 competition 평가 summary를 읽어 Kaggle Test 추론과
8개 object-row 컬럼 제출 파일을 생성한다.

핵심 계약:
- 모델 학습 데이터: 12,196 images / 46,394 objects / 118 classes
- 고정 split: Train 9,763 / Validation 2,433 / independent labeled Test 없음
- Kaggle Test: 842 images
- competition metric: mAP@[0.75:0.95]
- 모델 선택 근거: `week2/공통파이프라인/reports/eval_only_*_summary.json`
- submission format: `annotation_id,image_id,category_id,bbox_x,bbox_y,bbox_w,bbox_h,score`

> 원본 Kaggle `dataset.tar` 안의 train_images 232장은 **모델 학습 Train 수가 아니라**
> Kaggle 원본 archive 무결성 확인용이다. 현재 모델 Train 9,763장과 혼동하지 않는다.


## 먼저 준비할 입력

1. YOLO11s / YOLO11m / YOLO12m V4 평가가 모두 완료되어 있어야 한다.
   - `eval_only_yolo11s_*_summary.json`
   - `eval_only_yolo11m_*_summary.json`
   - `eval_only_yolo12m_*_summary.json`
2. 각 summary가 가리키는 completed checkpoint와 manifest가
   `week2/공통파이프라인/checkpoints`, `manifests`에 존재해야 한다.
3. Kaggle Test 842장이 포함된 원본 `dataset.tar`
4. `sample_submission.csv`는 있으면 자동 검증한다.

처음에는 `RUN_MODE="smoke"`로 8장만 확인한다.
정상 확인 후 `RUN_MODE="full"`로 바꿔 전체 842장 CSV를 생성한다.


#1. 기본세팅

##1-1. 재현 가능한 패키지 설치

GPU 메모리 단편화를 줄이는 설정을 PyTorch import 전에 적용하고, 학습 checkpoint와 호환되는 패키지 버전을 설치한다.


## 1-2. 라이브러리·Drive·환경 정보

In [16]:
# Colab 기본 PyTorch/CUDA 조합은 유지하고 필요한 상위 패키지만 고정한다.
import os
import subprocess
import sys

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

print("[Pipeline 01/14 |   7%] START: Package installation")

PINNED_PACKAGES = [
    "ultralytics==8.4.116",
    "tqdm==4.67.1",
]

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", *PINNED_PACKAGES
])

print("Package installation completed.")
print("PyTorch/torchvision were NOT force-downgraded; Colab runtime CUDA compatibility is preserved.")
print("[Pipeline 01/14 |   7%] DONE: Package installation")


[Pipeline 01/14 |   7%] START: Package installation
Package installation completed.
PyTorch/torchvision were NOT force-downgraded; Colab runtime CUDA compatibility is preserved.
[Pipeline 01/14 |   7%] DONE: Package installation


In [2]:
# 파일 검증, 이미지 전처리, 모델 추론과 결과 저장에 필요한 라이브러리를 불러온다.
import gc
import hashlib
import importlib.metadata as importlib_metadata
import json
import math
import os
import random
import re
import shutil
import sys  # [추가] RUNTIME_VERSIONS에서 사용하는 sys 모듈 누락분 반영
import time
import tarfile
from datetime import datetime
from io import BytesIO
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torchvision
from PIL import Image, ImageCms, ImageDraw, ImageOps
# [삭제] FRCNN용 불필요한 torch.utils.data 및 torchvision.transforms 임포트 제거
from tqdm.auto import tqdm
from ultralytics import YOLO


PIPELINE_STAGE_TOTAL = 14


def show_stage_progress(stage, title, status):
    """주요 셀의 전체 파이프라인 진행률을 같은 형식으로 표시한다."""
    percent = int(round(stage / PIPELINE_STAGE_TOTAL * 100))
    print(
        f"[Pipeline {stage:02d}/{PIPELINE_STAGE_TOTAL} | {percent:3d}%] "
        f"{status}: {title}"
    )


show_stage_progress(2, "Runtime, libraries, and Google Drive", "START")

try:
    from google.colab import drive
except ImportError as error:
    raise RuntimeError("This notebook must be executed in Google Colab.") from error

drive.mount("/content/drive")

# 실행할 때마다 같은 입력 순서와 결과가 나오도록 난수를 고정한다.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RUNTIME_VERSIONS = {
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "ultralytics": importlib_metadata.version("ultralytics"),
    "opencv": cv2.__version__,
    "pillow": importlib_metadata.version("Pillow"),
    "pandas": pd.__version__,
    "numpy": np.__version__,
}

print(f"Device: {DEVICE}")
print(json.dumps(RUNTIME_VERSIONS, indent=2))
show_stage_progress(2, "Runtime, libraries, and Google Drive", "DONE")

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
[Pipeline 02/14 |  14%] START: Runtime, libraries, and Google Drive
Mounted at /content/drive
Device: cuda
{
  "python": "3.12.13",
  "torch": "2.11.0+cu128",
  "torchvision": "0.26.0+cu128",
  "ultralytics": "8.4.116",
  "opencv": "4.14.0",
  "pillow": "11.3.0",
  "pandas": "2.2.3",
  "numpy": "2.0.2"
}
[Pipeline 02/14 |  14%] DONE: Runtime, libraries, and Google Drive


##1-3. 공통 설정과 경로

처음에는 `smoke`를 유지한다. `full`에서는 ***YOLO11s / YOLO11m / YOLO12m*** 제출 파일을 모두 만든다. 기본 `dataset.tar` 위치가 다르면 `DRIVE_DATASET_TAR_CANDIDATES`만 수정한다.


In [3]:
show_stage_progress(3, "V6.0.5 configuration and paths", "START")

# 빠른 연결 점검은 smoke, 실제 제출 생성은 full.
RUN_MODE = "full"  # "smoke" 또는 "full"
SMOKE_IMAGE_LIMIT = 8

CANDIDATE_MODELS = ("YOLO11s", "YOLO11m", "YOLO12m")
MODELS_TO_SUBMIT = CANDIDATE_MODELS

# full 제출 전 세 모델의 competition 평가 summary가 모두 있어야 한다.
REQUIRE_ALL_THREE_EVAL_SUMMARIES = True
EVAL_SUMMARY_OVERRIDES = {
    "YOLO11s": None,
    "YOLO11m": None,
    "YOLO12m": None,
}
SAMPLE_SUBMISSION_PATH_OVERRIDE = None
USE_TTA = False  # Validation에서 이득이 확인되기 전까지 False

ENABLE_INFERENCE_CACHE = True

COMPETITION_METRIC = "mAP@[0.75:0.95]"
KAGGLE_SUBMISSION_COLUMNS = [
    "annotation_id", "image_id", "category_id",
    "bbox_x", "bbox_y", "bbox_w", "bbox_h", "score",
]

PIPELINE_VERSION = "2.1-v605-current-fixed"

# ---------------------------------------------------------
# 서로 다른 두 계약을 분리해서 관리한다.
# ---------------------------------------------------------
# A. Kaggle 원본 dataset.tar 계약: Test 추출용 archive 자체의 무결성 확인
EXPECTED_BASE_DATASET_TRAIN_IMAGES = 232
EXPECTED_KAGGLE_TEST_IMAGES = 842

# B. 현재 V6.0.5 모델 학습/평가 계약
EXPECTED_MODEL_TOTAL_IMAGES = 12196
EXPECTED_MODEL_OBJECTS = 46394
EXPECTED_MODEL_CLASSES = 118
EXPECTED_MODEL_GROUPS = 4104
EXPECTED_MODEL_TRAIN_IMAGES = 9763
EXPECTED_MODEL_VAL_IMAGES = 2433
EXPECTED_CLASSES = EXPECTED_MODEL_CLASSES
EXPECTED_SPLIT_FINGERPRINT_PREFIX = "cc5d16d3fe042c52"

IMAGE_SIZE = 960
RAW_PREDICTION_CONFIDENCE = 0.001
NMS_IOU_THRESHOLD = 0.70
MAX_DETECTIONS = 300
MAX_PILLS_PER_IMAGE = None
YOLO_INFERENCE_BATCH_SIZE = 8
VISUALIZATION_IMAGE_COUNT = 4
SUPPORTED_IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"
}

if RUN_MODE not in {"smoke", "full"}:
    raise ValueError("RUN_MODE must be 'smoke' or 'full'.")
if tuple(MODELS_TO_SUBMIT) != CANDIDATE_MODELS:
    raise ValueError("MODELS_TO_SUBMIT must contain YOLO11s, YOLO11m, YOLO12m in order.")
if RUN_MODE == "full" and not torch.cuda.is_available():
    raise RuntimeError("Full Kaggle inference requires a GPU runtime.")

# ---------------------------------------------------------
# Drive paths
# ---------------------------------------------------------
BASE_ROOT = Path("/content/drive/MyDrive/baby_kangaroo")
SHARED_ROOT = BASE_ROOT / "week2"

PREPROCESS_ROOT = (
    SHARED_ROOT
    / "데이터전처리"
    / "yolo_전처리"
    / "pill_yolo_full_v6_0_preprocessed"
)
ARTIFACT_ROOT = SHARED_ROOT / "공통파이프라인"
CHECKPOINT_DIR = ARTIFACT_ROOT / "checkpoints"
MODEL_MANIFEST_DIR = ARTIFACT_ROOT / "manifests"
EVAL_REPORT_DIR = ARTIFACT_ROOT / "reports"

# 세 모델은 동일한 V6.0.5 class_mapping을 사용한다.
YOLO_BUNDLE_DIRS = {
    model_name: PREPROCESS_ROOT
    for model_name in CANDIDATE_MODELS
}

# Kaggle 원본 dataset.tar는 기존 공용 cache에서 가져온다.
COMMON_CACHE_ROOT = BASE_ROOT / "baby_kangaroo_cache"
DRIVE_KAGGLE_ROOT = SHARED_ROOT / "kaggle_submission_v2"

DATASET_TAR_OVERRIDE = os.getenv("KAGGLE_DATASET_TAR", "").strip()
DRIVE_DATASET_TAR_CANDIDATES = [
    Path(DATASET_TAR_OVERRIDE) if DATASET_TAR_OVERRIDE else None,
    COMMON_CACHE_ROOT / "dataset.tar",
    BASE_ROOT / "데이터전처리" / "yolo_전처리" / "dataset.tar",
    SHARED_ROOT / "데이터전처리" / "yolo_전처리" / "dataset.tar",
]
DRIVE_DATASET_TAR_CANDIDATES = [
    path for path in DRIVE_DATASET_TAR_CANDIDATES if path is not None
]
PROJECT_DIR_NAME = "sprint_ai_project1_data"

LOCAL_WORK_ROOT = Path("/content/baby_kangaroo_kaggle_submission_v21")
LOCAL_INPUT_ROOT = LOCAL_WORK_ROOT / "input"
LOCAL_MODEL_INPUT_DIR = LOCAL_WORK_ROOT / "model_input"
LOCAL_RESULT_ROOT = LOCAL_WORK_ROOT / "results"
KAGGLE_TEST_IMAGE_DIR = LOCAL_INPUT_ROOT / "test_images"

FINAL_CONFIG_DIR = LOCAL_RESULT_ROOT / "final_config"
PREPROCESSED_ROOT = LOCAL_RESULT_ROOT / "preprocessed_test"
PREDICTION_DIR = LOCAL_RESULT_ROOT / "predictions"
SUBMISSION_DIR = LOCAL_RESULT_ROOT / "submissions"
FIGURE_DIR = LOCAL_RESULT_ROOT / "figures"
REPORT_DIR = LOCAL_RESULT_ROOT / "reports"

DRIVE_FINAL_CONFIG_DIR = DRIVE_KAGGLE_ROOT / "final_config"
DRIVE_PREDICTION_DIR = DRIVE_KAGGLE_ROOT / "predictions"
DRIVE_SUBMISSION_DIR = DRIVE_KAGGLE_ROOT / "submissions"
DRIVE_FIGURE_DIR = DRIVE_KAGGLE_ROOT / "figures"
DRIVE_REPORT_DIR = DRIVE_KAGGLE_ROOT / "reports"
DRIVE_PREPROCESSING_MANIFEST_DIR = DRIVE_KAGGLE_ROOT / "preprocessing_manifests"

for directory in [
    LOCAL_INPUT_ROOT, LOCAL_MODEL_INPUT_DIR,
    FINAL_CONFIG_DIR, PREPROCESSED_ROOT, PREDICTION_DIR,
    SUBMISSION_DIR, FIGURE_DIR, REPORT_DIR,
    DRIVE_FINAL_CONFIG_DIR, DRIVE_PREDICTION_DIR,
    DRIVE_SUBMISSION_DIR, DRIVE_FIGURE_DIR,
    DRIVE_REPORT_DIR, DRIVE_PREPROCESSING_MANIFEST_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

# 현재 V6.0.5 핵심 리소스는 시작 단계에서 바로 확인한다.
for required_path in [
    PREPROCESS_ROOT / "class_mapping.csv",
    PREPROCESS_ROOT / "dataset_manifest.csv",
    CHECKPOINT_DIR,
    MODEL_MANIFEST_DIR,
    EVAL_REPORT_DIR,
]:
    if not required_path.exists():
        raise FileNotFoundError(f"Required V6.0.5 resource missing: {required_path}")

pipeline_timer_started_at = datetime.now()
pipeline_timer_started_perf = time.perf_counter()

print(f"Run mode: {RUN_MODE}")
print(f"Candidate models: {list(CANDIDATE_MODELS)}")
print(f"V6.0.5 preprocessing root: {PREPROCESS_ROOT}")
print(f"Evaluation report dir: {EVAL_REPORT_DIR}")
print(f"Checkpoint dir: {CHECKPOINT_DIR}")
print(f"Kaggle output dir: {DRIVE_KAGGLE_ROOT}")
print(f"Dataset.tar candidates: {[str(p) for p in DRIVE_DATASET_TAR_CANDIDATES]}")
print(f"Metric: {COMPETITION_METRIC} / raw conf={RAW_PREDICTION_CONFIDENCE} / NMS={NMS_IOU_THRESHOLD}")

show_stage_progress(3, "V6.0.5 configuration and paths", "DONE")


[Pipeline 03/14 |  21%] START: V6.0.5 configuration and paths
Run mode: full
Candidate models: ['YOLO11s', 'YOLO11m', 'YOLO12m']
V6.0.5 preprocessing root: /content/drive/MyDrive/baby_kangaroo/week2/데이터전처리/yolo_전처리/pill_yolo_full_v6_0_preprocessed
Evaluation report dir: /content/drive/MyDrive/baby_kangaroo/week2/공통파이프라인/reports
Checkpoint dir: /content/drive/MyDrive/baby_kangaroo/week2/공통파이프라인/checkpoints
Kaggle output dir: /content/drive/MyDrive/baby_kangaroo/week2/kaggle_submission_v2
Dataset.tar candidates: ['/content/drive/MyDrive/baby_kangaroo/baby_kangaroo_cache/dataset.tar', '/content/drive/MyDrive/baby_kangaroo/데이터전처리/yolo_전처리/dataset.tar', '/content/drive/MyDrive/baby_kangaroo/week2/데이터전처리/yolo_전처리/dataset.tar']
Metric: mAP@[0.75:0.95] / raw conf=0.001 / NMS=0.7
[Pipeline 03/14 |  21%] DONE: V6.0.5 configuration and paths


**V2.1 CURRENT FIXED:** 현재 V6.0.5 공통 데이터·체크포인트·평가 summary 계약을 사용합니다.


#2. 모델개발 결과에서 최종 설정 불러오기

이 단계는 모델을 다시 비교하거나 최고 모델 하나만 고르지 않는다. 모델개발 파이프라인의 `standard` 보고서에서
 ***YOLO11s / YOLO11m / YOLO12m***각각의 선택된 checkpoint와 Validation confidence를 읽고, 파일 해시를 확인해 세 모델 추론 계약으로 고정한다.


In [4]:
show_stage_progress(4, "Reusable validation and storage helpers", "START")

def sha256_file(path, chunk_size=1024 * 1024):
    """파일을 메모리에 한꺼번에 올리지 않고 SHA256 지문을 계산한다."""
    digest = hashlib.sha256()
    with Path(path).open("rb") as file:
        while True:
            chunk = file.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def stable_json_hash(value):
    """딕셔너리 순서와 관계없이 같은 JSON 내용에 같은 SHA256 지문을 만든다."""
    encoded = json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    ).encode("utf-8")
    return hashlib.sha256(encoded).hexdigest()


def json_ready(value):
    """Path·NumPy 자료형을 JSON으로 저장 가능한 기본 자료형으로 바꾼다."""
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, dict):
        return {str(key): json_ready(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(item) for item in value]
    if isinstance(value, np.generic):
        return value.item()
    return value


def write_json_atomic(path, value):
    """완성되지 않은 JSON이 남지 않도록 임시 파일을 거쳐 원자적으로 저장한다."""
    path = Path(path)
    temporary_path = path.with_suffix(path.suffix + ".tmp")
    temporary_path.write_text(
        json.dumps(json_ready(value), ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    temporary_path.replace(path)


def find_latest_standard_report(report_dir, expected_pipeline_version=None):
    """가장 최근에 완주한 standard 모델개발 보고서를 찾고 파이프라인 버전 호환성을 종합 검증한다."""
    candidates = sorted(
        Path(report_dir).glob("model_development_report_*.json"),
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )
    for path in candidates:
        try:
            report = json.loads(path.read_text(encoding="utf-8"))
        except (OSError, json.JSONDecodeError):
            continue
        if report.get("run_profile") == "standard":
            # 요구사항 4: pipeline version 및 계약 구성 요소 호환성 검증
            if expected_pipeline_version and report.get("pipeline_version") != expected_pipeline_version:
                continue
            return path, report

    raise FileNotFoundError(
        "No completed standard model development report matching the required pipeline version was found. "
        "Run the model development notebook in standard mode first."
    )

def find_checkpoint_manifest(checkpoint_path, model_name=None):
    """선택된 checkpoint 경로/모델 이름에 대응하는 완료 manifest를 찾아 해시와 내용을 엄격히 대조 검증한다."""
    checkpoint_path = Path(checkpoint_path)
    actual_hash = sha256_file(checkpoint_path)

    # 모델명이 직접 주어진 경우 또는 파일명에서 베이스 이름을 추출
    if model_name:
        target_prefix = model_name
    else:
        target_prefix = re.sub(r"(_best|_last)?\.(pth|pt)$", "", checkpoint_path.name)

    # 앙상블 대응: 단일 파일 검색 및 하위 디렉터리 재귀 검색 지원
    candidates = sorted(MODEL_MANIFEST_DIR.rglob(f"*{target_prefix}*.json"))

    # 정확히 매칭되는 유효 매니페스트 필터링
    valid_candidates = []
    for cand in candidates:
        try:
            m_data = json.loads(cand.read_text(encoding="utf-8"))
            if m_data.get("model_name") == model_name or target_prefix in cand.name:
                valid_candidates.append((cand, m_data))
        except (OSError, json.JSONDecodeError):
            continue

    if not valid_candidates:
        raise FileNotFoundError(
            f"Expected checkpoint manifest for {checkpoint_path.name} (model: {model_name}), but found none in {MODEL_MANIFEST_DIR}."
        )

    # 요구사항 2: 체크포인트 파일 해시와 1:1로 정확히 일치하는 Manifest를 우선 매칭 검증
    matched_pair = None
    for cand, m_data in valid_candidates:
        expected_hash = m_data.get("checkpoint_sha256") or m_data.get("weights_sha256")
        if expected_hash and expected_hash == actual_hash:
            matched_pair = (cand, m_data)
            break

    # 해시 일치 항목이 없는 경우 최신 수정된 파일 기준으로 폴백 후 아래에서 해시 오류 발생
    if matched_pair is None:
        manifest_path, manifest = sorted(valid_candidates, key=lambda x: x[0].stat().st_mtime, reverse=True)[0]
    else:
        manifest_path, manifest = matched_pair

    # 요구사항 1: training_completed 필드가 명시적으로 True인지 엄격 확인 (키 누락 시 통과 방지)
    if manifest.get("training_completed") is not True:
        raise RuntimeError(f"Training not completed or manifest incomplete for {checkpoint_path}")

    # 요구사항 2 최종 대조: 체크포인트 SHA256 불일치 차단
    expected_hash = manifest.get("checkpoint_sha256") or manifest.get("weights_sha256")
    if expected_hash and expected_hash != actual_hash:
        raise RuntimeError(f"Checkpoint SHA256 mismatch for {checkpoint_path.name}: expected {expected_hash}, got {actual_hash}")

    return manifest_path, manifest, actual_hash


def list_supported_images(image_dir):
    """지원 확장자의 이미지를 재귀적으로 수집한다."""
    image_dir = Path(image_dir)
    if not image_dir.is_dir():
        raise FileNotFoundError(f"Image directory not found: {image_dir}")
    return sorted(
        path for path in image_dir.rglob("*")
        if path.is_file() and path.suffix.lower() in SUPPORTED_IMAGE_EXTENSIONS
    )

def validate_test_image_list(image_paths, source_name):
    """Kaggle Test 이미지 수와 파일명 충돌을 동적으로 검사한다."""
    actual_count = len(image_paths)

    if hasattr(sys.modules[__name__], "EXPECTED_KAGGLE_TEST_IMAGES"):
        expected_count = EXPECTED_KAGGLE_TEST_IMAGES
        if expected_count is not None and actual_count != expected_count:
            raise RuntimeError(
                f"Expected {expected_count} Kaggle test images in {source_name}, "
                f"found {actual_count}."
            )

    # 이미지가 0개일 때만 에러 발생 (동적 검증의 최소 안전장치)
    if actual_count == 0:
        raise RuntimeError(f"No test images found in {source_name}.")

    basenames = [path.name for path in image_paths]
    stems = [path.stem for path in image_paths]
    if len(basenames) != len(set(basenames)):
        raise RuntimeError(f"Duplicate test image basenames were found in {source_name}.")
    if len(stems) != len(set(stems)):
        raise RuntimeError(
            f"Test images with identical stems and different extensions were found in {source_name}."
        )
    return image_paths


def reset_local_owned_directory(directory):
    """이 노트북이 소유한 /content 하위 폴더만 안전하게 초기화한다."""
    directory = Path(directory).resolve()
    owned_root = LOCAL_WORK_ROOT.resolve()
    if directory == owned_root or owned_root not in directory.parents:
        raise RuntimeError(f"Local directory reset is not allowed: {directory}")
    if directory.exists():
        shutil.rmtree(directory)
    directory.mkdir(parents=True, exist_ok=True)


def copy_file_verified(source, destination, expected_sha256=None):
    """파일을 임시 이름으로 복사한 뒤 크기·선택적 SHA256을 확인해 교체한다."""
    source = Path(source)
    destination = Path(destination)
    if not source.is_file():
        raise FileNotFoundError(f"Source file not found: {source}")
    destination.parent.mkdir(parents=True, exist_ok=True)

    source_stat = source.stat()
    reusable = destination.is_file() and destination.stat().st_size == source_stat.st_size
    if reusable and expected_sha256 is not None:
        reusable = sha256_file(destination) == expected_sha256
    elif reusable:
        reusable = int(destination.stat().st_mtime) == int(source_stat.st_mtime)
    if reusable:
        print(f"Reused verified local file: {destination}")
        return destination

    temporary_path = destination.with_suffix(destination.suffix + ".copying")
    shutil.copy2(source, temporary_path)
    if temporary_path.stat().st_size != source_stat.st_size:
        temporary_path.unlink(missing_ok=True)
        raise RuntimeError(f"Copied file size mismatch: {source} -> {destination}")
    if expected_sha256 is not None and sha256_file(temporary_path) != expected_sha256:
        temporary_path.unlink(missing_ok=True)
        raise RuntimeError(f"Copied file SHA256 mismatch: {source} -> {destination}")
    temporary_path.replace(destination)
    print(f"Copied to Colab local storage: {source.name}")
    return destination


def safe_extract_tar(archive_path, destination):
    """경로 이탈·특수 파일·디스크 부족을 차단한 뒤 TAR을 /content에 해제한다."""
    archive_path = Path(archive_path)
    destination = Path(destination)
    reset_local_owned_directory(destination)
    destination_root = destination.resolve()

    with tarfile.open(archive_path, "r:*") as archive:
        members = archive.getmembers()
        total_uncompressed_bytes = sum(member.size for member in members if member.isfile())
        available_bytes = shutil.disk_usage(LOCAL_WORK_ROOT).free
        if total_uncompressed_bytes > available_bytes * 0.80:
            raise RuntimeError(
                "The base dataset archive is too large for the available Colab local disk."
            )
        for member in members:
            target = (destination_root / member.name).resolve()
            if target != destination_root and destination_root not in target.parents:
                raise RuntimeError(f"Unsafe TAR member path: {member.name}")
            if member.issym() or member.islnk() or member.isdev():
                raise RuntimeError(f"Unsupported TAR member type: {member.name}")
        archive.extractall(destination_root)


def find_base_dataset_project(extract_root):
    """dataset.tar에서 프로젝트 폴더 하나만 찾고 수량을 동적으로 검사한다."""
    candidates = sorted({
        path.resolve()
        for path in Path(extract_root).rglob(PROJECT_DIR_NAME)
        if path.is_dir()
    })
    if len(candidates) != 1:
        raise RuntimeError(
            f"Expected exactly one {PROJECT_DIR_NAME} directory, found {candidates}."
        )
    project_root = candidates[0]
    train_image_dir = project_root / "train_images"
    test_image_dir = project_root / "test_images"
    annotation_dir = project_root / "train_annotations"
    missing = [
        path for path in [train_image_dir, test_image_dir, annotation_dir]
        if not path.is_dir()
    ]
    if missing:
        raise FileNotFoundError(f"Base dataset directories missing: {missing}")

    train_images = list_supported_images(train_image_dir)
    test_images = validate_test_image_list(
        list_supported_images(test_image_dir),
        str(test_image_dir),
    )

    # 동적 검증
    if len(train_images) != EXPECTED_BASE_DATASET_TRAIN_IMAGES:
        raise RuntimeError(
            f"Base Kaggle dataset.tar contract mismatch: expected "
            f"{EXPECTED_BASE_DATASET_TRAIN_IMAGES} train images, found {len(train_images)}."
        )

    return project_root, train_image_dir, test_image_dir, annotation_dir

def stage_kaggle_inputs_to_local():
    """기본 dataset.tar을 /content로 옮기고 Train 및 kaggle Test 데이터 계약을 확인한다."""
    dataset_tar_source = next(
        (path for path in DRIVE_DATASET_TAR_CANDIDATES if path.is_file()),
        None,
    )
    if dataset_tar_source is None:
        checked = "\n".join(f"- {path}" for path in DRIVE_DATASET_TAR_CANDIDATES)
        raise FileNotFoundError(
            f"Base dataset.tar was not found. Checked paths:\n{checked}"
        )

    local_dataset_tar = LOCAL_INPUT_ROOT / "dataset.tar"

    # 요구사항 3: 원본 dataset.tar 복사 시 SHA256 해시 계약을 엄격히 사전 검증하도록 expected_sha256 전달
    source_tar_sha256 = sha256_file(dataset_tar_source)
    copy_file_verified(dataset_tar_source, local_dataset_tar, expected_sha256=source_tar_sha256)

    archive_contract = {
        "source_path": str(dataset_tar_source),
        "source_size_bytes": dataset_tar_source.stat().st_size,
        "source_mtime": int(dataset_tar_source.stat().st_mtime),
        "local_archive_sha256": source_tar_sha256,
    }
    extract_root = LOCAL_INPUT_ROOT / "base_dataset"
    marker_path = LOCAL_INPUT_ROOT / "base_dataset_staging_contract.json"
    reuse_extracted = False
    if marker_path.is_file() and extract_root.is_dir():
        try:
            saved_contract = json.loads(marker_path.read_text(encoding="utf-8"))
            if saved_contract == archive_contract:
                project_root, train_image_dir, test_image_dir, annotation_dir = (
                    find_base_dataset_project(extract_root)
                )
                reuse_extracted = True
        except (OSError, json.JSONDecodeError, RuntimeError, FileNotFoundError):
            reuse_extracted = False

    if reuse_extracted:
        print(f"Reused local base dataset: {project_root}")
    else:
        safe_extract_tar(local_dataset_tar, extract_root)
        project_root, train_image_dir, test_image_dir, annotation_dir = (
            find_base_dataset_project(extract_root)
        )
        write_json_atomic(marker_path, archive_contract)

    test_images = validate_test_image_list(
        list_supported_images(test_image_dir),
        str(test_image_dir),
    )
    staging_rows = [
        {
            "file_name": path.name,
            "local_path": str(path),
            "size_bytes": path.stat().st_size,
            "staging_mode": "base_dataset_tar_to_colab_local",
            "drive_source": str(dataset_tar_source),
            "base_dataset_sha256": archive_contract["local_archive_sha256"],
        }
        for path in test_images
    ]
    return test_image_dir, "base_dataset_tar_to_colab_local", pd.DataFrame(staging_rows)

def persist_artifact_to_drive(local_path, drive_directory):
    """작은 최종 산출물만 Drive에 복사하고 SHA256 동일성을 확인한다."""
    local_path = Path(local_path)
    drive_directory = Path(drive_directory)
    drive_directory.mkdir(parents=True, exist_ok=True)
    destination = drive_directory / local_path.name
    expected_hash = sha256_file(local_path)
    return copy_file_verified(local_path, destination, expected_sha256=expected_hash)

show_stage_progress(4, "Reusable validation and storage helpers", "DONE")

[Pipeline 04/14 |  29%] START: Reusable validation and storage helpers
[Pipeline 04/14 |  29%] DONE: Reusable validation and storage helpers


In [5]:
show_stage_progress(5, "Load V6.0.5 competition evaluation summaries", "START")

MODEL_PREFIXES = {
    "YOLO11s": "yolo11s",
    "YOLO11m": "yolo11m",
    "YOLO12m": "yolo12m",
}

def find_latest_eval_summary(model_name):
    override = EVAL_SUMMARY_OVERRIDES.get(model_name)
    if override:
        path = Path(override)
        if not path.is_file():
            raise FileNotFoundError(f"{model_name} summary override not found: {path}")
        return path

    prefix = MODEL_PREFIXES[model_name]
    candidates = sorted(
        EVAL_REPORT_DIR.glob(
            f"eval_only_{prefix}_{EXPECTED_SPLIT_FINGERPRINT_PREFIX}_*_summary.json"
        ),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    return candidates[0] if candidates else None


def validate_eval_summary(model_name, summary_path, summary):
    required = {
        "mode", "training_disabled", "model_name",
        "competition_metric", "split_policy", "split_fingerprint",
        "train_images", "validation_images", "independent_test_available",
        "selected_stage", "selected_experiment_id",
        "selected_checkpoint_path", "selected_confidence",
        "selected_top_k", "selected_competition_mAP",
    }
    missing = required - set(summary)
    if missing:
        raise KeyError(f"{model_name} summary missing keys: {sorted(missing)}")

    if summary["mode"] != "evaluation_only" or summary["training_disabled"] is not True:
        raise RuntimeError(f"{model_name} summary is not the evaluation-only contract.")
    if summary["model_name"] != model_name:
        raise RuntimeError(
            f"Summary model mismatch: expected {model_name}, got {summary['model_name']}"
        )
    if summary["competition_metric"] != COMPETITION_METRIC:
        raise RuntimeError(
            f"{model_name} metric mismatch: {summary['competition_metric']}"
        )
    if summary["split_policy"] != "preserve_exact_upstream":
        raise RuntimeError(f"{model_name} uses unexpected split policy.")
    if not str(summary["split_fingerprint"]).startswith(
        EXPECTED_SPLIT_FINGERPRINT_PREFIX
    ):
        raise RuntimeError(
            f"{model_name} split fingerprint mismatch: {summary['split_fingerprint']}"
        )
    if int(summary["train_images"]) != EXPECTED_MODEL_TRAIN_IMAGES:
        raise RuntimeError(f"{model_name} Train count mismatch.")
    if int(summary["validation_images"]) != EXPECTED_MODEL_VAL_IMAGES:
        raise RuntimeError(f"{model_name} Validation count mismatch.")
    if summary["independent_test_available"] is not False:
        raise RuntimeError(f"{model_name} unexpectedly reports an independent labeled Test.")

    confidence = float(summary["selected_confidence"])
    if not 0.0 <= confidence <= 1.0:
        raise RuntimeError(f"{model_name} invalid selected confidence: {confidence}")
    top_k = summary["selected_top_k"]
    if top_k is not None and int(top_k) <= 0:
        raise RuntimeError(f"{model_name} invalid selected Top-K: {top_k}")

    return True


MODEL_RUN_CONFIGS = {}
EVAL_SUMMARIES = {}
selection_rows = []
missing_summaries = []

for model_name in CANDIDATE_MODELS:
    summary_path = find_latest_eval_summary(model_name)
    if summary_path is None:
        missing_summaries.append(model_name)
        continue

    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    validate_eval_summary(model_name, summary_path, summary)

    experiment_id = str(summary["selected_experiment_id"])
    expected_checkpoint_name = f"{experiment_id}_best.pt"
    checkpoint_name_from_summary = Path(summary["selected_checkpoint_path"]).name
    if checkpoint_name_from_summary != expected_checkpoint_name:
        raise RuntimeError(
            f"{model_name} summary checkpoint name mismatch: "
            f"{checkpoint_name_from_summary} != {expected_checkpoint_name}"
        )

    # summary 안의 절대경로를 그대로 믿지 않고 현재 공통 checkpoint 폴더에서 다시 찾는다.
    source_checkpoint_path = CHECKPOINT_DIR / expected_checkpoint_name
    if not source_checkpoint_path.is_file():
        raise FileNotFoundError(
            f"{model_name} selected checkpoint missing: {source_checkpoint_path}"
        )

    source_manifest_path = MODEL_MANIFEST_DIR / f"{experiment_id}.json"
    if not source_manifest_path.is_file():
        raise FileNotFoundError(
            f"{model_name} selected checkpoint manifest missing: {source_manifest_path}"
        )
    manifest = json.loads(source_manifest_path.read_text(encoding="utf-8"))

    if manifest.get("training_completed") is not True:
        raise RuntimeError(f"{model_name} checkpoint is not marked training_completed.")
    if manifest.get("model_name") != model_name:
        raise RuntimeError(
            f"{model_name} manifest model mismatch: {manifest.get('model_name')}"
        )
    if manifest.get("split_fingerprint") != summary["split_fingerprint"]:
        raise RuntimeError(f"{model_name} summary/manifest split fingerprint mismatch.")

    split_config = manifest.get("split_config", {})
    if int(split_config.get("expected_images", -1)) != EXPECTED_MODEL_TOTAL_IMAGES:
        raise RuntimeError(f"{model_name} manifest total image contract mismatch.")
    if int(split_config.get("expected_objects", -1)) != EXPECTED_MODEL_OBJECTS:
        raise RuntimeError(f"{model_name} manifest object contract mismatch.")
    if int(split_config.get("expected_num_classes", -1)) != EXPECTED_MODEL_CLASSES:
        raise RuntimeError(f"{model_name} manifest class contract mismatch.")
    if int(split_config.get("expected_groups", -1)) != EXPECTED_MODEL_GROUPS:
        raise RuntimeError(f"{model_name} manifest group contract mismatch.")

    checkpoint_sha256 = sha256_file(source_checkpoint_path)
    expected_hash = manifest.get("checkpoint_sha256")
    if expected_hash and checkpoint_sha256 != expected_hash:
        raise RuntimeError(
            f"{model_name} checkpoint SHA256 mismatch: "
            f"expected {expected_hash}, got {checkpoint_sha256}"
        )

    # 재현을 위해 summary / manifest / checkpoint를 Colab 로컬에 검증 복사한다.
    local_summary_path = LOCAL_MODEL_INPUT_DIR / f"{model_name}_{summary_path.name}"
    local_manifest_path = LOCAL_MODEL_INPUT_DIR / f"{model_name}_{source_manifest_path.name}"
    local_checkpoint_path = (
        LOCAL_MODEL_INPUT_DIR
        / f"{model_name}_{checkpoint_sha256[:12]}_{source_checkpoint_path.name}"
    )
    copy_file_verified(
        summary_path,
        local_summary_path,
        expected_sha256=sha256_file(summary_path),
    )
    copy_file_verified(
        source_manifest_path,
        local_manifest_path,
        expected_sha256=sha256_file(source_manifest_path),
    )
    copy_file_verified(
        source_checkpoint_path,
        local_checkpoint_path,
        expected_sha256=checkpoint_sha256,
    )

    selected_top_k = (
        None
        if summary["selected_top_k"] is None
        else int(summary["selected_top_k"])
    )

    MODEL_RUN_CONFIGS[model_name] = {
        "model_name": model_name,
        "experiment_id": experiment_id,
        "selected_stage": str(summary["selected_stage"]),
        "source_summary_path": summary_path,
        "local_summary_path": local_summary_path,
        "source_checkpoint_path": source_checkpoint_path,
        "local_checkpoint_path": local_checkpoint_path,
        "checkpoint_sha256": checkpoint_sha256,
        "source_manifest_path": source_manifest_path,
        "local_manifest_path": local_manifest_path,
        "selected_confidence": float(summary["selected_confidence"]),
        "selected_top_k": selected_top_k,
        "selected_competition_mAP": float(summary["selected_competition_mAP"]),
        "dataset_fingerprint": manifest.get("dataset_fingerprint"),
        "split_fingerprint": manifest.get("split_fingerprint"),
    }
    EVAL_SUMMARIES[model_name] = summary

    selection_rows.append({
        "model": model_name,
        "stage": summary["selected_stage"],
        "experiment": experiment_id,
        "competition_mAP": float(summary["selected_competition_mAP"]),
        "confidence": float(summary["selected_confidence"]),
        "top_k": selected_top_k,
        "checkpoint": source_checkpoint_path.name,
        "summary": summary_path.name,
    })

if missing_summaries and REQUIRE_ALL_THREE_EVAL_SUMMARIES:
    raise FileNotFoundError(
        "Competition evaluation summary is still missing for: "
        + ", ".join(missing_summaries)
        + ". Finish the corresponding V4 evaluation first, then rerun this notebook."
    )

if not MODEL_RUN_CONFIGS:
    raise RuntimeError("No validated competition evaluation summaries were found.")

# 세 모델 summary가 존재하면 split/dataset 계약도 서로 같아야 한다.
split_fingerprints = {
    config["split_fingerprint"] for config in MODEL_RUN_CONFIGS.values()
}
dataset_fingerprints = {
    config["dataset_fingerprint"] for config in MODEL_RUN_CONFIGS.values()
}
if len(split_fingerprints) != 1:
    raise RuntimeError(f"Model split fingerprints disagree: {split_fingerprints}")
if len(dataset_fingerprints) != 1:
    raise RuntimeError(f"Model dataset fingerprints disagree: {dataset_fingerprints}")

MODEL_SELECTION_TABLE = (
    pd.DataFrame(selection_rows)
    .sort_values(["competition_mAP", "model"], ascending=[False, True])
    .reset_index(drop=True)
)

RECOMMENDED_MODEL = str(MODEL_SELECTION_TABLE.iloc[0]["model"])
RECOMMENDED_COMPETITION_MAP = float(
    MODEL_SELECTION_TABLE.iloc[0]["competition_mAP"]
)

display(MODEL_SELECTION_TABLE)
print(
    f"RECOMMENDED SINGLE MODEL: {RECOMMENDED_MODEL} / "
    f"{COMPETITION_METRIC}={RECOMMENDED_COMPETITION_MAP:.6f}"
)

three_model_selection_contract = {
    "pipeline_version": PIPELINE_VERSION,
    "created_at": datetime.now().isoformat(),
    "selection_source": (
        "Latest V6.0.5 eval_only summaries on the fixed upstream Validation"
    ),
    "models_to_submit": list(MODEL_RUN_CONFIGS),
    "recommended_model": RECOMMENDED_MODEL,
    "recommended_competition_mAP": RECOMMENDED_COMPETITION_MAP,
    "ranking": MODEL_SELECTION_TABLE.to_dict("records"),
    "models": MODEL_RUN_CONFIGS,
    "image_size": IMAGE_SIZE,
    "raw_prediction_confidence": RAW_PREDICTION_CONFIDENCE,
    "nms_iou_threshold": NMS_IOU_THRESHOLD,
    "max_detections": MAX_DETECTIONS,
    "dataset_fingerprint": next(iter(dataset_fingerprints)),
    "split_fingerprint": next(iter(split_fingerprints)),
    "model_contract": {
        "total_images": EXPECTED_MODEL_TOTAL_IMAGES,
        "objects": EXPECTED_MODEL_OBJECTS,
        "classes": EXPECTED_MODEL_CLASSES,
        "groups": EXPECTED_MODEL_GROUPS,
        "train_images": EXPECTED_MODEL_TRAIN_IMAGES,
        "validation_images": EXPECTED_MODEL_VAL_IMAGES,
        "independent_test_available": False,
    },
    "kaggle_test_images": EXPECTED_KAGGLE_TEST_IMAGES,
}

selection_signature = stable_json_hash(three_model_selection_contract)[:16]
FINAL_CONFIG_PATH = FINAL_CONFIG_DIR / f"v605_submission_config_{selection_signature}.json"
write_json_atomic(FINAL_CONFIG_PATH, three_model_selection_contract)

print(f"V6.0.5 submission contract: {FINAL_CONFIG_PATH}")
show_stage_progress(5, "Load V6.0.5 competition evaluation summaries", "DONE")


[Pipeline 05/14 |  36%] START: Load V6.0.5 competition evaluation summaries
Copied to Colab local storage: eval_only_yolo11s_cc5d16d3fe042c52_20260819_010312_summary.json
Copied to Colab local storage: yolo11s_baseline_b29ddd0378fc1d32.json
Copied to Colab local storage: yolo11s_baseline_b29ddd0378fc1d32_best.pt
Copied to Colab local storage: eval_only_yolo11m_cc5d16d3fe042c52_20260819_012252_summary.json
Copied to Colab local storage: yolo11m_baseline_21cc7ae361705449.json
Copied to Colab local storage: yolo11m_baseline_21cc7ae361705449_best.pt
Copied to Colab local storage: eval_only_yolo12m_cc5d16d3fe042c52_20260819_015116_summary.json
Copied to Colab local storage: yolo12m_baseline_c38729a7d52bad35.json
Copied to Colab local storage: yolo12m_baseline_c38729a7d52bad35_best.pt


,model,stage,experiment,competition_mAP,confidence,top_k,checkpoint,summary
0,YOLO11m,baseline,yolo11m_baseline_21cc7ae361705449,0.973039,0.001,6,yolo11m_baseline_21cc7ae361705449_best.pt,eval_only_yolo11m_cc5d16d3fe042c52_20260819_01...
1,YOLO12m,baseline,yolo12m_baseline_c38729a7d52bad35,0.972983,0.001,12,yolo12m_baseline_c38729a7d52bad35_best.pt,eval_only_yolo12m_cc5d16d3fe042c52_20260819_01...
2,YOLO11s,baseline,yolo11s_baseline_b29ddd0378fc1d32,0.971988,0.001,6,yolo11s_baseline_b29ddd0378fc1d32_best.pt,eval_only_yolo11s_cc5d16d3fe042c52_20260819_01...


RECOMMENDED SINGLE MODEL: YOLO11m / mAP@[0.75:0.95]=0.973039
V6.0.5 submission contract: /content/baby_kangaroo_kaggle_submission_v21/results/final_config/v605_submission_config_8380680e9a2ed00c.json
[Pipeline 05/14 |  36%] DONE: Load V6.0.5 competition evaluation summaries


**V2.1 CURRENT FIXED:** 현재 V6.0.5 공통 데이터·체크포인트·평가 summary 계약을 사용합니다.


#3. Kaggle 입력 파일 확인

(기본 Test 이미지는 새 파일이 아니라 전처리에서 사용한 동일한 기본 데이터셋의 `test_images/`다.-X)

***기본 Test 이미지는 별도의 외부 파일이 아니라 전처리 단계에서 공유하여 사용하는 동일한 기본 데이터셋의 test_images/ 폴더 내 데이터를 사용합니다.***

이 셀은
Drive의 `/content/drive/MyDrive/

baby_kangaroo_cache/dataset.tar`을 `/content`로 한 번 복사하고, `sprint_ai_project1_data/test_images/`를 자동으로 찾는다.

별도 제출 양식 파일은 필요하지 않다. 대회 안내에 따라 Test 이미지 파일명의 숫자를 `image_id`로 사용하고, 제출용 ID 변환표를 자동으로 만든다.

(기본 데이터 계약은 Train 및 Test 데이터 구조를 확인하며, 파일명, stem과 변환된 `image_id` 충돌도 검사해 잘못된 이미지와 예측이 연결되는 일을 막는다.-x)

***기본 Test 이미지는 별도의 외부 파일이 아니라 전처리 단계에서 공유하여 사용하는 동일한 기본 데이터셋의 test_images/ 폴더 내 데이터를 사용합니다.***


In [6]:
show_stage_progress(6, "Stage and validate Kaggle test inputs", "START")

def collect_test_images(image_dir):
    """Kaggle Test 전체 이미지를 수집하고 파일명 충돌을 검사한다."""
    return validate_test_image_list(
        list_supported_images(image_dir),
        str(image_dir),
    )


def parse_competition_image_id(file_name):
    """파일명의 숫자를 대회 image_id 정수로 변환하되 애매한 이름은 중단한다."""
    stem = Path(file_name).stem
    if stem.isdigit():
        image_id = int(stem)
    else:
        number_groups = re.findall(r"\d+", stem)
        if len(number_groups) != 1:
            raise ValueError(
                "The test filename must contain exactly one unambiguous numeric image ID: "
                f"{file_name!r}"
            )
        image_id = int(number_groups[0])
    if image_id < 1:
        raise ValueError(f"Kaggle image_id must be positive: {file_name!r}")
    return image_id


def build_test_image_id_mapping(image_paths):
    """전체 파일명과 대회 image_id 사이의 일대일 대응표를 만든다."""
    mapping_df = pd.DataFrame({
        "file_name": [path.name for path in image_paths],
        "image_id": [parse_competition_image_id(path.name) for path in image_paths],
    })
    if mapping_df["file_name"].duplicated().any():
        raise RuntimeError("Duplicate test filenames were found in the image ID mapping.")
    if mapping_df["image_id"].duplicated().any():
        duplicates = mapping_df.loc[
            mapping_df["image_id"].duplicated(keep=False),
            ["file_name", "image_id"],
        ]
        raise RuntimeError(
            "Multiple test files map to the same Kaggle image_id: "
            f"{duplicates.head(10).to_dict('records')}"
        )
    return mapping_df.sort_values("image_id").reset_index(drop=True)


KAGGLE_TEST_IMAGE_DIR, input_staging_mode, input_staging_manifest_df = (
    stage_kaggle_inputs_to_local()
)
LOCAL_STAGING_MANIFEST_PATH = REPORT_DIR / "local_input_staging_manifest.csv"
input_staging_manifest_df.to_csv(
    LOCAL_STAGING_MANIFEST_PATH,
    index=False,
    encoding="utf-8-sig",
)

all_test_image_paths = collect_test_images(KAGGLE_TEST_IMAGE_DIR)
test_image_id_mapping_df = build_test_image_id_mapping(all_test_image_paths)
TEST_IMAGE_ID_MAPPING_PATH = REPORT_DIR / "test_image_id_mapping.csv"
test_image_id_mapping_df.to_csv(
    TEST_IMAGE_ID_MAPPING_PATH,
    index=False,
    encoding="utf-8-sig",
)

active_test_image_paths = (
    all_test_image_paths[:SMOKE_IMAGE_LIMIT]
    if RUN_MODE == "smoke"
    else all_test_image_paths
)

print(f"Input staging mode: {input_staging_mode}")
print(f"Local Kaggle test root: {KAGGLE_TEST_IMAGE_DIR}")
print(f"Kaggle test images: {len(all_test_image_paths)}")
print(f"Unique Kaggle image IDs: {test_image_id_mapping_df['image_id'].nunique()}")
print(f"Images used in this run: {len(active_test_image_paths)}")
print(f"Image ID mapping: {TEST_IMAGE_ID_MAPPING_PATH}")
display(test_image_id_mapping_df.head())

show_stage_progress(6, "Stage and validate Kaggle test inputs", "DONE")

[Pipeline 06/14 |  43%] START: Stage and validate Kaggle test inputs
Copied to Colab local storage: dataset.tar


/tmp/ipykernel_3358/131006946.py:229: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  archive.extractall(destination_root)


Input staging mode: base_dataset_tar_to_colab_local
Local Kaggle test root: /content/baby_kangaroo_kaggle_submission_v21/input/base_dataset/dataset/sprint_ai_project1_data/test_images
Kaggle test images: 842
Unique Kaggle image IDs: 842
Images used in this run: 842
Image ID mapping: /content/baby_kangaroo_kaggle_submission_v21/results/reports/test_image_id_mapping.csv


,file_name,image_id
0,1.png,1
1,3.png,3
2,4.png,4
3,5.png,5
4,8.png,8


[Pipeline 06/14 |  43%] DONE: Stage and validate Kaggle test inputs


In [7]:
# V2: sample_submission.csv가 있으면 공식 schema와 test image_id 집합을 직접 대조한다.
def locate_sample_submission():
    candidates = [
        Path(SAMPLE_SUBMISSION_PATH_OVERRIDE) if SAMPLE_SUBMISSION_PATH_OVERRIDE else None,
        LOCAL_INPUT_ROOT / "sample_submission.csv",
        Path("/content/sample_submission.csv"),
        SHARED_ROOT / "sample_submission.csv",
    ]
    return next((path for path in candidates if path is not None and path.is_file()), None)

SAMPLE_SUBMISSION_PATH = locate_sample_submission()
if SAMPLE_SUBMISSION_PATH is None:
    print("sample_submission.csv not found: object-row schema remains code-validated; image_id audit uses test filenames.")
else:
    sample_submission_df = pd.read_csv(SAMPLE_SUBMISSION_PATH)
    if sample_submission_df.columns.tolist() != KAGGLE_SUBMISSION_COLUMNS:
        raise RuntimeError(
            f"sample_submission schema mismatch: {sample_submission_df.columns.tolist()}"
        )
    sample_ids = set(pd.to_numeric(sample_submission_df["image_id"], errors="coerce").dropna().astype(int))
    mapped_ids = set(test_image_id_mapping_df["image_id"].astype(int))
    # sample이 모든 test image를 row로 표현하는 형식일 때만 집합 일치를 강제한다.
    if sample_ids and len(sample_ids) >= len(mapped_ids) and sample_ids != mapped_ids:
        raise RuntimeError(
            f"sample_submission image_id mismatch: missing={sorted(mapped_ids-sample_ids)[:10]}, "
            f"extra={sorted(sample_ids-mapped_ids)[:10]}"
        )
    print(f"sample_submission audit PASS: {SAMPLE_SUBMISSION_PATH}")


sample_submission.csv not found: object-row schema remains code-validated; image_id audit uses test filenames.


#4. Kaggle Test 이미지 전처리

학습 데이터와 같은 순서로 EXIF·ICC를 정규화하고, bilateral filter, 제한된 gray-world 화이트밸런스, LAB L 채널 CLAHE, 960x960 letterbox를 적용한다. 증강은 적용하지 않는다.

각 이미지의 원본 크기, scale과 padding을 manifest에 저장한다. 추론 후 BBox를 원본 좌표로 되돌릴 때 이 값만 사용한다.

In [8]:
show_stage_progress(7, "Define deterministic v6.0 preprocessing", "START")

# v6.0 학습 데이터와 동일한 결정적 전처리 설정이다.
PREPROCESS_TARGET_SIZE = 960
PREPROCESS_DENOISE_DIAMETER = 5
PREPROCESS_DENOISE_SIGMA_COLOR = 20.0
PREPROCESS_DENOISE_SIGMA_SPACE = 20.0
PREPROCESS_WHITE_BALANCE_GAIN_LIMIT = 0.05
PREPROCESS_CLAHE_CLIP_LIMIT = 1.5
PREPROCESS_CLAHE_GRID = (8, 8)

PREPROCESSING_CONFIG = {
    "order": [
        "EXIF orientation normalization",
        "ICC-aware sRGB conversion",
        "edge-preserving bilateral denoising",
        "clipped gray-world white balance",
        "LAB luminance CLAHE",
        "aspect-ratio-preserving letterbox resize",
    ],
    "target_size": [PREPROCESS_TARGET_SIZE, PREPROCESS_TARGET_SIZE],
    "denoise": {
        "method": "cv2.bilateralFilter",
        "diameter": PREPROCESS_DENOISE_DIAMETER,
        "sigma_color": PREPROCESS_DENOISE_SIGMA_COLOR,
        "sigma_space": PREPROCESS_DENOISE_SIGMA_SPACE,
    },
    "color_correction": {
        "white_balance": "gray_world_with_clipped_channel_gains",
        "gain_limit": PREPROCESS_WHITE_BALANCE_GAIN_LIMIT,
        "luminance_equalization": "CLAHE_on_LAB_L_only",
        "clahe_clip_limit": PREPROCESS_CLAHE_CLIP_LIMIT,
        "clahe_grid": list(PREPROCESS_CLAHE_GRID),
    },
    "letterbox_padding": "median RGB of the corrected image border",
    "augmentation": "disabled_for_kaggle_inference",
}

if PREPROCESS_TARGET_SIZE != IMAGE_SIZE:
    raise RuntimeError("Kaggle preprocessing size must match the model input size.")


def image_to_srgb(image):
    """EXIF 방향과 ICC 프로필을 반영해 안전한 RGB 이미지로 변환한다."""
    normalized = ImageOps.exif_transpose(image)
    icc_profile = normalized.info.get("icc_profile")
    if "A" in normalized.getbands() or "transparency" in normalized.info:
        rgba = normalized.convert("RGBA")
        background = Image.new("RGB", rgba.size, "white")
        background.paste(rgba, mask=rgba.getchannel("A"))
        normalized = background
    else:
        normalized = normalized.convert("RGB")
    if icc_profile:
        try:
            source_profile = ImageCms.ImageCmsProfile(BytesIO(icc_profile))
            target_profile = ImageCms.createProfile("sRGB")
            normalized = ImageCms.profileToProfile(
                normalized,
                source_profile,
                target_profile,
                outputMode="RGB",
                renderingIntent=0,
            )
        except Exception as error:
            raise ValueError(f"ICC profile conversion failed: {error}") from error
    return normalized


def border_median_rgb(rgb_array, border_fraction=0.04):
    """letterbox 여백에 사용할 이미지 테두리의 중앙 RGB 값을 계산한다."""
    height, width = rgb_array.shape[:2]
    thickness = max(1, int(round(min(height, width) * border_fraction)))
    border_pixels = np.concatenate([
        rgb_array[:thickness].reshape(-1, 3),
        rgb_array[-thickness:].reshape(-1, 3),
        rgb_array[:, :thickness].reshape(-1, 3),
        rgb_array[:, -thickness:].reshape(-1, 3),
    ], axis=0)
    return np.median(border_pixels, axis=0).round().clip(0, 255).astype(np.uint8)


def clipped_gray_world_white_balance(rgb_array):
    """채널별 gain을 제한해 과보정 없이 gray-world 화이트밸런스를 적용한다."""
    work = rgb_array.astype(np.float32)
    channel_means = work.reshape(-1, 3).mean(axis=0)
    gray_mean = float(channel_means.mean())
    raw_gains = gray_mean / np.maximum(channel_means, 1.0)
    low = 1.0 - PREPROCESS_WHITE_BALANCE_GAIN_LIMIT
    high = 1.0 + PREPROCESS_WHITE_BALANCE_GAIN_LIMIT
    gains = np.clip(raw_gains, low, high)
    balanced = np.clip(work * gains.reshape(1, 1, 3), 0, 255).astype(np.uint8)
    return balanced, gains


def clahe_luminance_only(rgb_array):
    """색상 채널은 유지하고 LAB의 밝기 채널에만 CLAHE를 적용한다."""
    lab = cv2.cvtColor(rgb_array, cv2.COLOR_RGB2LAB)
    l_channel, a_channel, b_channel = cv2.split(lab)
    clahe = cv2.createCLAHE(
        clipLimit=PREPROCESS_CLAHE_CLIP_LIMIT,
        tileGridSize=PREPROCESS_CLAHE_GRID,
    )
    enhanced_l = clahe.apply(l_channel)
    return cv2.cvtColor(
        cv2.merge((enhanced_l, a_channel, b_channel)),
        cv2.COLOR_LAB2RGB,
    )


def save_rgb_image(rgb_array, destination):
    """원본 확장자에 맞춰 전처리 RGB 이미지를 저장한다."""
    destination = Path(destination)
    image = Image.fromarray(rgb_array, mode="RGB")
    suffix = destination.suffix.lower()
    if suffix in {".jpg", ".jpeg"}:
        image.save(destination, format="JPEG", quality=95, subsampling=0)
    elif suffix == ".png":
        image.save(destination, format="PNG", compress_level=3)
    else:
        image.save(destination)


def preprocess_kaggle_image(source, destination):
    """학습 데이터와 같은 결정적 전처리를 적용하고 역변환 정보를 반환한다."""
    source = Path(source)
    destination = Path(destination)
    with Image.open(source) as raw_image:
        rgb = np.asarray(image_to_srgb(raw_image), dtype=np.uint8)

    source_height, source_width = rgb.shape[:2]
    if source_width <= 0 or source_height <= 0:
        raise ValueError(f"Invalid image size: {source}")

    denoised = cv2.bilateralFilter(
        rgb,
        d=PREPROCESS_DENOISE_DIAMETER,
        sigmaColor=PREPROCESS_DENOISE_SIGMA_COLOR,
        sigmaSpace=PREPROCESS_DENOISE_SIGMA_SPACE,
    )
    white_balanced, wb_gains = clipped_gray_world_white_balance(denoised)
    corrected = clahe_luminance_only(white_balanced)

    nominal_scale = min(
        PREPROCESS_TARGET_SIZE / source_width,
        PREPROCESS_TARGET_SIZE / source_height,
    )
    resized_width = max(1, int(round(source_width * nominal_scale)))
    resized_height = max(1, int(round(source_height * nominal_scale)))
    interpolation = cv2.INTER_AREA if nominal_scale < 1.0 else cv2.INTER_LANCZOS4
    resized = cv2.resize(corrected, (resized_width, resized_height), interpolation=interpolation)

    pad_left = (PREPROCESS_TARGET_SIZE - resized_width) // 2
    pad_top = (PREPROCESS_TARGET_SIZE - resized_height) // 2
    padding_rgb = border_median_rgb(corrected)
    canvas = np.empty((PREPROCESS_TARGET_SIZE, PREPROCESS_TARGET_SIZE, 3), dtype=np.uint8)
    canvas[...] = padding_rgb
    canvas[pad_top:pad_top + resized_height, pad_left:pad_left + resized_width] = resized
    save_rgb_image(canvas, destination)

    # [수정] 전처리 결과 파일 저장 후 SHA256 해시 생성 (캐시 무결성 검증용)
    processed_sha256 = sha256_file(destination)

    return {
        "file_name": source.name,
        "source_width": int(source_width),
        "source_height": int(source_height),
        "output_width": PREPROCESS_TARGET_SIZE,
        "output_height": PREPROCESS_TARGET_SIZE,
        "resized_width": int(resized_width),
        "resized_height": int(resized_height),
        "scale_x": float(resized_width / source_width),
        "scale_y": float(resized_height / source_height),
        "pad_left": int(pad_left),
        "pad_top": int(pad_top),
        "padding_r": int(padding_rgb[0]),
        "padding_g": int(padding_rgb[1]),
        "padding_b": int(padding_rgb[2]),
        "wb_gain_r": float(wb_gains[0]),
        "wb_gain_g": float(wb_gains[1]),
        "wb_gain_b": float(wb_gains[2]),
        "processed_sha256": processed_sha256,  # [추가] 캐시 재사용 검증을 위한 SHA256 필드
    }

show_stage_progress(7, "Define deterministic v6.0 preprocessing", "DONE")

[Pipeline 07/14 |  50%] START: Define deterministic v6.0 preprocessing
[Pipeline 07/14 |  50%] DONE: Define deterministic v6.0 preprocessing


In [9]:
show_stage_progress(8, "Preprocess and cache active test images", "START")

# Kaggle 입력 해시·캐시 확인·전처리에 걸린 시간을 함께 측정한다.
preprocessing_started_at = datetime.now()
preprocessing_started_perf = time.perf_counter()

# 활성 이미지의 파일 지문을 이용해 동일한 전처리 결과만 재사용한다.
active_inventory = []
for source_path in tqdm(active_test_image_paths, desc="Hash Kaggle test images", unit="image"):
    active_inventory.append({
        "file_name": source_path.name,
        "source_path": str(source_path),
        "source_sha256": sha256_file(source_path),
        "source_size_bytes": source_path.stat().st_size,
    })

preprocessing_contract = {
    "run_mode": RUN_MODE,
    "config": PREPROCESSING_CONFIG,
    "images": active_inventory,
}
preprocessing_signature = stable_json_hash(preprocessing_contract)[:16]
ACTIVE_PREPROCESSED_DIR = PREPROCESSED_ROOT / preprocessing_signature / "images"
PREPROCESSING_MANIFEST_PATH = (
    PREPROCESSED_ROOT / preprocessing_signature / "preprocessing_manifest.csv"
)
ACTIVE_PREPROCESSED_DIR.mkdir(parents=True, exist_ok=True)

reuse_preprocessing = False
if PREPROCESSING_MANIFEST_PATH.is_file():
    cached_manifest = pd.read_csv(PREPROCESSING_MANIFEST_PATH)

    # [수정된 부분 시작] -------------------------------------------------------
    # 단순 파일 존재 여부뿐만 아니라, manifest에 기록된 processed_sha256과
    # 실제 파일의 해시값이 일치하는지 무결성을 검증하는 로직을 추가했습니다.
    if "processed_sha256" in cached_manifest.columns:
        reuse_preprocessing = (
            len(cached_manifest) == len(active_test_image_paths)
            and cached_manifest["file_name"].is_unique
            and all(
                (ACTIVE_PREPROCESSED_DIR / row["file_name"]).is_file() and
                sha256_file(ACTIVE_PREPROCESSED_DIR / row["file_name"]) == row["processed_sha256"]
                for _, row in cached_manifest.iterrows()
            )
        )
    else:
        reuse_preprocessing = False
    # [수정된 부분 끝] ---------------------------------------------------------

if reuse_preprocessing:
    preprocessing_manifest_df = cached_manifest
    print(f"Reused verified Kaggle preprocessing cache: {PREPROCESSING_MANIFEST_PATH}")
else:
    inventory_by_name = {row["file_name"]: row for row in active_inventory}
    transform_rows = []
    for source_path in tqdm(
        active_test_image_paths,
        desc="Preprocess Kaggle test images",
        unit="image",
    ):
        destination_path = ACTIVE_PREPROCESSED_DIR / source_path.name
        transform = preprocess_kaggle_image(source_path, destination_path)
        transform_rows.append({
            **transform,
            "source_path": str(source_path),
            "processed_path": str(destination_path),
            "source_sha256": inventory_by_name[source_path.name]["source_sha256"],
            "processed_sha256": sha256_file(destination_path),
            "preprocessing_signature": preprocessing_signature,
        })
    preprocessing_manifest_df = pd.DataFrame(transform_rows).sort_values("file_name")
    preprocessing_manifest_df.to_csv(
        PREPROCESSING_MANIFEST_PATH,
        index=False,
        encoding="utf-8-sig",
    )

if len(preprocessing_manifest_df) != len(active_test_image_paths):
    raise RuntimeError("Kaggle preprocessing manifest count mismatch.")
if not (
    (preprocessing_manifest_df["output_width"] == IMAGE_SIZE).all()
    and (preprocessing_manifest_df["output_height"] == IMAGE_SIZE).all()
):
    raise RuntimeError("A preprocessed Kaggle image is not 960x960.")

print(f"Preprocessed images: {len(preprocessing_manifest_df)}")
print(f"Preprocessing manifest: {PREPROCESSING_MANIFEST_PATH}")

preprocessing_seconds = time.perf_counter() - preprocessing_started_perf
preprocessing_finished_at = datetime.now()
print(f"Preprocessing seconds: {preprocessing_seconds:.2f}")

show_stage_progress(8, "Preprocess and cache active test images", "DONE")

[Pipeline 08/14 |  57%] START: Preprocess and cache active test images


Hash Kaggle test images:   0%|          | 0/842 [00:00<?, ?image/s]

Preprocess Kaggle test images:   0%|          | 0/842 [00:00<?, ?image/s]

/tmp/ipykernel_3358/2012616618.py:114: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  image = Image.fromarray(rgb_array, mode="RGB")


Preprocessed images: 842
Preprocessing manifest: /content/baby_kangaroo_kaggle_submission_v21/results/preprocessed_test/c19d2288912a96f9/preprocessing_manifest.csv
Preprocessing seconds: 253.12
[Pipeline 08/14 |  57%] DONE: Preprocess and cache active test images


#5. 선택된 checkpoint 불러오기

세 모델의 v6.0 클래스 매핑을 모두 읽고 검증한다. 실제 모델은 동시에 올리지 않으며, 추론 순서가 왔을 때 하나만 GPU에 로드하고 완료 즉시 해제한다.


**+수정사항 설명**

표준 모델 Class Mapping 및 GPU 메모리 관리
* 세 모델(YOLO11s, YOLO11m, YOLO12m)의 v6.0 클래스 매핑을 모두 읽고 검증합니다.
* 실제 모델은 동시에 올리지 않으며, 추론 순서가 왔을 때 하나만 GPU에 로드하고 완료 즉시 해제합니다.

In [10]:
show_stage_progress(9, "Validate mappings and define memory-safe model loaders", "START")


def load_class_mapping(model_name):
    """모델별 class_mapping.csv를 검증해 원본 category ID 복원표를 만든다."""
    source_mapping_path = YOLO_BUNDLE_DIRS[model_name] / "class_mapping.csv"
    model_label_column = "yolo_class_id"
    class_name_candidates = ["normalized_class_name", "class_name", "original_class_name"]

    if not source_mapping_path.is_file():
        raise FileNotFoundError(f"Class mapping not found: {source_mapping_path}")
    mapping_path = LOCAL_MODEL_INPUT_DIR / f"{model_name}_class_mapping.csv"
    copy_file_verified(
        source_mapping_path,
        mapping_path,
        expected_sha256=sha256_file(source_mapping_path),
    )
    mapping_df = pd.read_csv(mapping_path)
    required = {model_label_column, "original_category_id"}
    missing = required - set(mapping_df.columns)
    if missing:
        raise ValueError(f"Class mapping columns missing: {sorted(missing)}")
    if len(mapping_df) != EXPECTED_CLASSES:
        raise RuntimeError(f"Expected {EXPECTED_CLASSES} classes, found {len(mapping_df)}.")

    mapping_df[model_label_column] = pd.to_numeric(
        mapping_df[model_label_column], errors="raise"
    ).astype(int)
    mapping_df["original_category_id"] = pd.to_numeric(
        mapping_df["original_category_id"], errors="raise"
    ).astype(int)
    if mapping_df[model_label_column].duplicated().any():
        raise RuntimeError(f"Duplicate {model_name} model labels were found.")
    if sorted(mapping_df[model_label_column].tolist()) != list(range(EXPECTED_CLASSES)):
        raise RuntimeError(
            f"{model_name} YOLO class IDs must be exactly 0..{EXPECTED_CLASSES - 1}."
        )
    if mapping_df["original_category_id"].duplicated().any():
        raise RuntimeError(f"Duplicate {model_name} original category IDs were found.")

    class_name_column = next(
        (column for column in class_name_candidates if column in mapping_df.columns),
        None,
    )
    model_to_original = dict(zip(
        mapping_df[model_label_column],
        mapping_df["original_category_id"],
    ))
    original_to_name = {
        int(row["original_category_id"]): (
            str(row[class_name_column]) if class_name_column else str(row["original_category_id"])
        )
        for _, row in mapping_df.iterrows()
    }
    return {
        "mapping_path": mapping_path,
        "mapping_df": mapping_df,
        "model_label_column": model_label_column,
        "model_to_original": model_to_original,
        "original_to_name": original_to_name,
        "allowed_original_category_ids": set(model_to_original.values()),
    }


def load_inference_bundle(model_name):
    """지정한 모델 하나만 메모리에 불러와 추론 context를 반환한다."""
    config = MODEL_RUN_CONFIGS[model_name]
    mapping = CLASS_MAPPING_CONTRACTS[model_name]
    checkpoint_path = Path(config["local_checkpoint_path"])

    model = YOLO(str(checkpoint_path))

    return {
        "model_name": model_name,
        "model": model,
        "config": config,
        "mapping": mapping,
    }


def release_inference_bundle(bundle):
    """완료된 모델을 CPU로 내리고 참조·CUDA 캐시를 정리한다."""
    model = bundle.pop("model", None) if bundle is not None else None
    if model is not None:
        try:
            if hasattr(model, "predictor"):
                model.predictor = None
            if hasattr(model, "model") and isinstance(model.model, torch.nn.Module):
                model.model.to("cpu")
            elif isinstance(model, torch.nn.Module):
                model.to("cpu")
        except Exception:
            pass
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except RuntimeError:
            pass


def current_gpu_memory_gb():
    """현재 PyTorch CUDA 메모리 사용량을 GB로 반환한다."""
    if not torch.cuda.is_available():
        return 0.0
    return float(torch.cuda.memory_allocated() / (1024 ** 3))


CLASS_MAPPING_CONTRACTS = {}
mapping_rows = []
for model_name in tqdm(MODELS_TO_SUBMIT, desc="Validate class mappings", unit="model"):
    mapping_contract = load_class_mapping(model_name)
    CLASS_MAPPING_CONTRACTS[model_name] = mapping_contract
    mapping_rows.append({
        "model": model_name,
        "classes": len(mapping_contract["mapping_df"]),
        "original_category_ids": len(mapping_contract["allowed_original_category_ids"]),
        "mapping_path": str(mapping_contract["mapping_path"]),
    })

display(pd.DataFrame(mapping_rows))
show_stage_progress(9, "Validate mappings and define memory-safe model loaders", "DONE")

[Pipeline 09/14 |  64%] START: Validate mappings and define memory-safe model loaders


Validate class mappings:   0%|          | 0/3 [00:00<?, ?model/s]

Copied to Colab local storage: class_mapping.csv
Copied to Colab local storage: class_mapping.csv
Copied to Colab local storage: class_mapping.csv


,model,classes,original_category_ids,mapping_path
0,YOLO11s,118,118,/content/baby_kangaroo_kaggle_submission_v21/m...
1,YOLO11m,118,118,/content/baby_kangaroo_kaggle_submission_v21/m...
2,YOLO12m,118,118,/content/baby_kangaroo_kaggle_submission_v21/m...


[Pipeline 09/14 |  64%] DONE: Validate mappings and define memory-safe model loaders


**+수정사항 설명**
1. FRCNN 전용 모델 빌더 함수(build_frcnn_for_inference) 및 PyTorch torchvision 기반 FRCNN 로딩 분기문이 통째로 삭제되었습니다.  

2. YOLO 전용 로더(load_inference_bundle, release_inference_bundle)가 YOLO11s, YOLO11m, YOLO12m을 범용적으로 로드/해제할 수 있도록 확장 적용되었습니다.  

- 가이드라인 부합 여부: 셀 23의 FRCNN loader 분기 삭제 및 YOLO 3종 확장 지침 준수.  

#6. 세 모델의 kaggle Test 추론과 원본 좌표 복원


모델 예측은 960x960 전처리 좌표다. 저장한 scale과 padding으로 원본 좌표를 복원하고 이미지 범위 안으로 제한한다. 세 모델은 같은 Test 이미지와 좌표 복원 규칙을 사용하지만, 각 모델의 Validation confidence와 checkpoint는 독립적으로 적용한다.

In [11]:
show_stage_progress(10, "Define three-model inference functions", "START")

PREDICTION_COLUMNS = [
    "file_name",
    "category_id",
    "score",
    "x_min",
    "y_min",
    "x_max",
    "y_max",
    "x",
    "y",
    "width",
    "height",
    "rank",
    "model",
]


def inverse_letterbox_box(box_xyxy, transform):
    """960x960 xyxy BBox를 원본 이미지 좌표로 복원하고 범위 안으로 자른다."""
    x_min, y_min, x_max, y_max = map(float, box_xyxy)
    x_min = (x_min - float(transform["pad_left"])) / float(transform["scale_x"])
    x_max = (x_max - float(transform["pad_left"])) / float(transform["scale_x"])
    y_min = (y_min - float(transform["pad_top"])) / float(transform["scale_y"])
    y_max = (y_max - float(transform["pad_top"])) / float(transform["scale_y"])

    source_width = float(transform["source_width"])
    source_height = float(transform["source_height"])
    x_min = float(np.clip(x_min, 0.0, source_width))
    x_max = float(np.clip(x_max, 0.0, source_width))
    y_min = float(np.clip(y_min, 0.0, source_height))
    y_max = float(np.clip(y_max, 0.0, source_height))
    if not all(math.isfinite(value) for value in [x_min, y_min, x_max, y_max]):
        raise ValueError("A restored BBox contains NaN or infinity.")
    if x_max <= x_min or y_max <= y_min:
        return None
    return [x_min, y_min, x_max, y_max]


def normalize_image_predictions(
    file_name,
    boxes,
    labels,
    scores,
    transform,
    model_name,
    selected_confidence,
    selected_top_k,
    model_to_original,
):
    """한 모델의 예측을 원본 category ID와 원본 좌표의 공통 행으로 변환한다."""
    candidates = []
    for box, label, score in zip(boxes, labels, scores):
        score = float(score)
        if score < selected_confidence:
            continue
        model_label = int(label)
        if model_label not in model_to_original:
            raise KeyError(f"Unknown {model_name} label {model_label} in {file_name}.")
        restored = inverse_letterbox_box(box, transform)
        if restored is None:
            continue
        x_min, y_min, x_max, y_max = restored
        candidates.append({
            "file_name": file_name,
            "category_id": int(model_to_original[model_label]),
            "score": score,
            "x_min": x_min,
            "y_min": y_min,
            "x_max": x_max,
            "y_max": y_max,
            "x": x_min,
            "y": y_min,
            "width": x_max - x_min,
            "height": y_max - y_min,
            "model": model_name,
        })

    candidates.sort(key=lambda row: row["score"], reverse=True)
    selected = candidates if selected_top_k is None else candidates[:int(selected_top_k)]
    for rank, row in enumerate(selected, start=1):
        row["rank"] = rank
    return selected


def run_yolo_kaggle_inference(bundle, manifest_df):
    """YOLO를 디렉터리 stream·batch configurable로 실행해 목록 일괄 GPU 로딩을 막는다."""
    model_name = bundle["model_name"]
    model = bundle["model"]
    config = bundle["config"]
    mapping = bundle["mapping"]
    transform_by_name = manifest_df.set_index("file_name").to_dict("index")
    processed_paths = [Path(path) for path in manifest_df["processed_path"].tolist()]
    processed_directories = {path.resolve().parent for path in processed_paths}
    if len(processed_directories) != 1:
        raise RuntimeError(f"{model_name} preprocessed images must share one directory.")
    processed_directory = next(iter(processed_directories))

    expected_names = set(manifest_df["file_name"])
    actual_names = {path.name for path in list_supported_images(processed_directory)}
    if actual_names != expected_names:
        raise RuntimeError(
            f"{model_name} streaming directory does not match the preprocessing manifest. "
            f"missing={sorted(expected_names - actual_names)[:5]}, "
            f"extra={sorted(actual_names - expected_names)[:5]}"
        )

    # [수정된 부분 시작] -------------------------------------------------------
    # USE_TTA가 문자열 'False'일 때 bool('False') == True 가 되어
    # 옵션이 무력화(항상 켜짐)되는 현상을 막기 위해 명시적으로 파싱합니다.
    # 또한 TTA가 켜지면 메모리 사용량이 급증하므로, batch 사이즈를 안전하게 1로 강제합니다.
    is_tta_enabled = (
        str(USE_TTA).strip().lower() in ["true", "1", "t", "y", "yes"]
        if isinstance(USE_TTA, str)
        else bool(USE_TTA)
    )

    predict_kwargs = {
        "source": str(processed_directory),
        "stream": True,
        "batch": 1 if is_tta_enabled else YOLO_INFERENCE_BATCH_SIZE,
        "imgsz": IMAGE_SIZE,
        "conf": RAW_PREDICTION_CONFIDENCE,
        "iou": NMS_IOU_THRESHOLD,
        "max_det": MAX_DETECTIONS,
        "device": 0 if torch.cuda.is_available() else "cpu",
        "verbose": False,
        "augment": is_tta_enabled,
    }
    # [수정된 부분 끝] ---------------------------------------------------------

    rows = []
    seen_names = set()
    ordered_expected_names = [path.name for path in processed_paths]
    results = model.predict(**predict_kwargs)
    for result_index, result in enumerate(tqdm(
        results,
        total=len(processed_paths),
        desc=f"{model_name} Kaggle inference",
        unit="image",
    )):
        returned_name = Path(str(getattr(result, "path", ""))).name
        file_name = returned_name
        if file_name not in transform_by_name:
            generic_name = bool(re.fullmatch(r"image\d+\.[A-Za-z0-9]+", file_name))
            if generic_name and result_index < len(ordered_expected_names):
                file_name = ordered_expected_names[result_index]
            else:
                raise KeyError(
                    f"{model_name} returned an unknown image name: {returned_name!r}"
                )
        if file_name in seen_names:
            raise RuntimeError(f"{model_name} returned a duplicate image result: {file_name}")
        seen_names.add(file_name)

        boxes = result.boxes
        if boxes is None or len(boxes) == 0:
            continue
        rows.extend(normalize_image_predictions(
            file_name,
            boxes.xyxy.detach().cpu().numpy(),
            boxes.cls.detach().cpu().numpy(),
            boxes.conf.detach().cpu().numpy(),
            transform_by_name[file_name],
            model_name,
            float(config["selected_confidence"]),
            config.get("selected_top_k"),
            mapping["model_to_original"],
        ))

    if seen_names != expected_names:
        raise RuntimeError(
            f"{model_name} result count mismatch: "
            f"expected={len(expected_names)}, returned={len(seen_names)}"
        )
    return rows


show_stage_progress(10, "Define three-model inference functions", "DONE")

[Pipeline 10/14 |  71%] START: Define three-model inference functions
[Pipeline 10/14 |  71%] DONE: Define three-model inference functions


**V2.1 CURRENT FIXED:** 현재 V6.0.5 공통 데이터·체크포인트·평가 summary 계약을 사용합니다.


In [12]:
show_stage_progress(11, "Run YOLO11s, YOLO11m, and YOLO12m inference", "START")


def build_inference_manifest(raw_predictions_df, preprocessing_manifest):
    """한 모델의 이미지별 최종 검출 개수를 포함한 추론 manifest를 만든다."""
    prediction_counts = (
        raw_predictions_df.groupby("file_name").size().to_dict()
        if not raw_predictions_df.empty
        else {}
    )
    manifest_df = preprocessing_manifest[[
        "file_name", "source_path", "source_width", "source_height"
    ]].copy()
    manifest_df["prediction_count"] = manifest_df["file_name"].map(
        prediction_counts
    ).fillna(0).astype(int)
    return manifest_df


def validate_model_inference_output(
    model_name,
    raw_predictions_df,
    inference_manifest_df,
    preprocessing_manifest,
    allowed_category_ids,
):
    """캐시 또는 새 추론 결과가 현재 입력·클래스 계약과 맞는지 확인한다."""
    if raw_predictions_df.columns.tolist() != PREDICTION_COLUMNS:
        raise RuntimeError(f"{model_name} raw prediction columns do not match the contract.")
    expected_names = set(preprocessing_manifest["file_name"])
    actual_names = set(inference_manifest_df["file_name"])
    if len(inference_manifest_df) != len(preprocessing_manifest) or actual_names != expected_names:
        raise RuntimeError(f"{model_name} inference manifest image contract failed.")
    if inference_manifest_df["file_name"].duplicated().any():
        raise RuntimeError(f"{model_name} inference manifest contains duplicate files.")

    # [수정] IndentationError 교정 및 MAX_PILLS_PER_IMAGE 조건 검사 안전화
    if MAX_PILLS_PER_IMAGE is not None:
        if (inference_manifest_df["prediction_count"] > MAX_PILLS_PER_IMAGE).any():
            raise ValueError(f"Image produced more than {MAX_PILLS_PER_IMAGE} pills")

    unknown_categories = set(
        raw_predictions_df.get("category_id", pd.Series(dtype=int)).astype(int)
    ) - set(map(int, allowed_category_ids))
    if unknown_categories:
        raise RuntimeError(f"{model_name} produced unknown category IDs: {unknown_categories}")
    if not raw_predictions_df.empty and set(raw_predictions_df["file_name"]) - expected_names:
        raise RuntimeError(f"{model_name} predictions contain unknown filenames.")


def execute_one_model_inference(model_name, preprocessing_manifest):
    """모델 하나를 추론하거나 검증된 로컬 캐시를 재사용한 뒤 결과만 반환한다."""
    config = MODEL_RUN_CONFIGS[model_name]
    mapping = CLASS_MAPPING_CONTRACTS[model_name]

    # [수정] inference_contract 서명 키에 selected_top_k, use_tta, max_detections, runtime_versions 추가
    inference_contract = {
        "pipeline_version": PIPELINE_VERSION,
        "run_mode": RUN_MODE,
        "model_name": model_name,
        "checkpoint_sha256": config["checkpoint_sha256"],
        "selected_confidence": config["selected_confidence"],
        "selected_top_k": config.get("selected_top_k"),
        "use_tta": USE_TTA,
        "max_detections": MAX_DETECTIONS,
        "preprocessing_signature": preprocessing_signature,
        "images": preprocessing_manifest["file_name"].tolist(),
        "image_size": IMAGE_SIZE,
        "raw_confidence": RAW_PREDICTION_CONFIDENCE,
        "nms_iou": NMS_IOU_THRESHOLD,
        "max_pills_per_image": MAX_PILLS_PER_IMAGE,
        "inference_batch_size": YOLO_INFERENCE_BATCH_SIZE,
        "runtime_versions": RUNTIME_VERSIONS,
    }
    inference_signature = stable_json_hash(inference_contract)[:16]
    raw_path = PREDICTION_DIR / f"raw_predictions_{model_name.lower()}_{inference_signature}.csv"
    manifest_path = PREDICTION_DIR / f"inference_manifest_{model_name.lower()}_{inference_signature}.csv"
    cache_marker_path = PREDICTION_DIR / f"inference_cache_{model_name.lower()}_{inference_signature}.json"

    if ENABLE_INFERENCE_CACHE and all(
        path.is_file() for path in [raw_path, manifest_path, cache_marker_path]
    ):
        try:
            cache_marker = json.loads(cache_marker_path.read_text(encoding="utf-8"))

            # [수정] 저장된 raw_predictions_sha256 및 inference_manifest_sha256과 실제 파일 해시 일치 검증
            raw_hash_match = cache_marker.get("raw_predictions_sha256") == sha256_file(raw_path)
            manifest_hash_match = cache_marker.get("inference_manifest_sha256") == sha256_file(manifest_path)

            if (
                cache_marker.get("inference_signature") == inference_signature
                and raw_hash_match
                and manifest_hash_match
            ):
                raw_predictions_df = pd.read_csv(raw_path)
                inference_manifest_df = pd.read_csv(manifest_path)
                validate_model_inference_output(
                    model_name,
                    raw_predictions_df,
                    inference_manifest_df,
                    preprocessing_manifest,
                    mapping["allowed_original_category_ids"],
                )
                print(f"Reused verified {model_name} inference cache: {raw_path}")
                return {
                    "model_name": model_name,
                    "raw_predictions_df": raw_predictions_df,
                    "inference_manifest_df": inference_manifest_df,
                    "raw_predictions_path": raw_path,
                    "inference_manifest_path": manifest_path,
                    "cache_marker_path": cache_marker_path,
                    "inference_signature": inference_signature,
                    "inference_seconds": float(cache_marker["inference_seconds"]),
                    "peak_gpu_memory_gb": float(cache_marker.get("peak_gpu_memory_gb", 0.0)),
                    "cache_reused": True,
                    "allowed_original_category_ids": sorted(
                        mapping["allowed_original_category_ids"]
                    ),
                }
        except (OSError, ValueError, KeyError, json.JSONDecodeError, RuntimeError):
            print(f"Ignored invalid {model_name} inference cache and reran inference.")

    release_inference_bundle(None)
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    bundle = None
    inference_started_wall_at = datetime.now()
    inference_started_perf = time.perf_counter()
    try:
        print(f"Loading {model_name}. GPU allocated before load: {current_gpu_memory_gb():.2f} GB")
        bundle = load_inference_bundle(model_name)
        prediction_rows = run_yolo_kaggle_inference(bundle, preprocessing_manifest)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        inference_seconds = time.perf_counter() - inference_started_perf
        peak_gpu_memory_gb = (
            float(torch.cuda.max_memory_allocated() / (1024 ** 3))
            if torch.cuda.is_available()
            else 0.0
        )
    finally:
        release_inference_bundle(bundle)

    raw_predictions_df = pd.DataFrame(prediction_rows, columns=PREDICTION_COLUMNS)
    if not raw_predictions_df.empty:
        raw_predictions_df = raw_predictions_df.sort_values(
            ["file_name", "rank"], ascending=[True, True]
        ).reset_index(drop=True)
    inference_manifest_df = build_inference_manifest(
        raw_predictions_df,
        preprocessing_manifest,
    )
    validate_model_inference_output(
        model_name,
        raw_predictions_df,
        inference_manifest_df,
        preprocessing_manifest,
        mapping["allowed_original_category_ids"],
    )

    raw_predictions_df.to_csv(raw_path, index=False, encoding="utf-8-sig")
    inference_manifest_df.to_csv(manifest_path, index=False, encoding="utf-8-sig")
    cache_marker = {
        "created_at": datetime.now().isoformat(),
        "inference_started_at": inference_started_wall_at.isoformat(),
        "inference_signature": inference_signature,
        "contract": inference_contract,
        "inference_seconds": inference_seconds,
        "peak_gpu_memory_gb": peak_gpu_memory_gb,
        "detected_objects": len(raw_predictions_df),
        "raw_predictions_sha256": sha256_file(raw_path),
        "inference_manifest_sha256": sha256_file(manifest_path),
    }
    write_json_atomic(cache_marker_path, cache_marker)

    print(
        f"{model_name} completed: images={len(inference_manifest_df)}, "
        f"objects={len(raw_predictions_df)}, seconds={inference_seconds:.2f}, "
        f"peak_gpu={peak_gpu_memory_gb:.2f} GB"
    )
    return {
        "model_name": model_name,
        "raw_predictions_df": raw_predictions_df,
        "inference_manifest_df": inference_manifest_df,
        "raw_predictions_path": raw_path,
        "inference_manifest_path": manifest_path,
        "cache_marker_path": cache_marker_path,
        "inference_signature": inference_signature,
        "inference_seconds": inference_seconds,
        "peak_gpu_memory_gb": peak_gpu_memory_gb,
        "cache_reused": False,
        "allowed_original_category_ids": sorted(mapping["allowed_original_category_ids"]),
    }


MODEL_INFERENCE_OUTPUTS = {}
for model_name in tqdm(MODELS_TO_SUBMIT, desc="Three-model Kaggle inference", unit="model"):
    MODEL_INFERENCE_OUTPUTS[model_name] = execute_one_model_inference(
        model_name,
        preprocessing_manifest_df,
    )

submission_batch_id = (
    f"{datetime.now().strftime('%Y%m%d_%H%M%S')}_"
    f"{selection_signature}_{preprocessing_signature}"
)
model_inference_timing_table = pd.DataFrame([
    {
        "model": model_name,
        "images": len(output["inference_manifest_df"]),
        "detected_objects": len(output["raw_predictions_df"]),
        "selected_confidence": MODEL_RUN_CONFIGS[model_name]["selected_confidence"],
        "selected_top_k": MODEL_RUN_CONFIGS[model_name].get("selected_top_k"),
        "inference_seconds": output["inference_seconds"],
        "inference_minutes": output["inference_seconds"] / 60.0,
        "peak_gpu_memory_gb": output["peak_gpu_memory_gb"],
        "cache_reused": output["cache_reused"],
    }
    for model_name, output in MODEL_INFERENCE_OUTPUTS.items()
])
display(model_inference_timing_table)
show_stage_progress(11, "Run YOLO11s, YOLO11m, and YOLO12m inference", "DONE")

[Pipeline 11/14 |  79%] START: Run YOLO11s, YOLO11m, and YOLO12m inference


Three-model Kaggle inference:   0%|          | 0/3 [00:00<?, ?model/s]

Loading YOLO11s. GPU allocated before load: 0.00 GB


YOLO11s Kaggle inference:   0%|          | 0/842 [00:00<?, ?image/s]

YOLO11s completed: images=842, objects=4388, seconds=33.74, peak_gpu=1.08 GB
Loading YOLO11m. GPU allocated before load: 0.03 GB


...

YOLO11m completed: images=842, objects=4228, seconds=34.88, peak_gpu=1.86 GB
Loading YOLO12m. GPU allocated before load: 0.03 GB


YOLO12m Kaggle inference:   0%|          | 0/842 [00:00<?, ?image/s]

YOLO12m completed: images=842, objects=4923, seconds=38.12, peak_gpu=2.52 GB


,model,images,detected_objects,selected_confidence,selected_top_k,inference_seconds,inference_minutes,peak_gpu_memory_gb,cache_reused
0,YOLO11s,842,4388,0.001,6,33.739083,0.562318,1.075649,False
1,YOLO11m,842,4228,0.001,6,34.884859,0.581414,1.856948,False
2,YOLO12m,842,4923,0.001,12,38.116497,0.635275,2.523273,False


[Pipeline 11/14 |  79%] DONE: Run YOLO11s, YOLO11m, and YOLO12m inference


**V2.1 CURRENT FIXED:** 현재 V6.0.5 공통 데이터·체크포인트·평가 summary 계약을 사용합니다.


#7. 예측 결과 시각화

원본 이미지 위에 모델별 BBox를 그려 padding 제거와 좌표 역변환을 확인한다. 같은 이미지에 대해 YOLO11s / YOLO11m / YOLO12m 미리보기를 각각 저장하므로 세 모델의 오탐·미탐 차이도 눈으로 비교할 수 있다.


In [13]:
show_stage_progress(12, "Create prediction previews for three models", "START")


def draw_prediction_preview(image_path, prediction_df):
    """원본 이미지에 category ID와 confidence를 표시한다."""
    with Image.open(image_path) as image:
        canvas = ImageOps.exif_transpose(image).convert("RGB")
    drawer = ImageDraw.Draw(canvas)
    line_width = max(2, round(min(canvas.size) / 300))
    for row in prediction_df.itertuples(index=False):
        box = [row.x_min, row.y_min, row.x_max, row.y_max]
        drawer.rectangle(box, outline=(255, 40, 40), width=line_width)
        label = f"{int(row.category_id)} {float(row.score):.2f}"
        text_position = (max(0, row.x_min), max(0, row.y_min - 14))
        drawer.text(text_position, label, fill=(255, 40, 40))
    return canvas


PREVIEW_PATHS_BY_MODEL = {}
preview_count = min(VISUALIZATION_IMAGE_COUNT, len(active_test_image_paths))
for model_name in tqdm(MODELS_TO_SUBMIT, desc="Create model previews", unit="model"):
    raw_predictions_df = MODEL_INFERENCE_OUTPUTS[model_name]["raw_predictions_df"]
    fig, axes = plt.subplots(preview_count, 1, figsize=(12, 5 * preview_count))
    if preview_count == 1:
        axes = [axes]
    for axis, image_path in zip(axes, active_test_image_paths[:preview_count]):
        image_predictions = raw_predictions_df[
            raw_predictions_df["file_name"] == image_path.name
        ]
        preview = draw_prediction_preview(image_path, image_predictions)
        axis.imshow(preview)
        axis.set_title(
            f"{model_name} | {image_path.name} | detections={len(image_predictions)}"
        )
        axis.axis("off")
    plt.tight_layout()
    preview_path = FIGURE_DIR / (
        f"kaggle_prediction_preview_{model_name.lower()}_{submission_batch_id}.png"
    )
    plt.savefig(preview_path, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    PREVIEW_PATHS_BY_MODEL[model_name] = preview_path
    print(f"{model_name} prediction preview: {preview_path}")

show_stage_progress(12, "Create prediction previews for three models", "DONE")


Output hidden; open in https://colab.research.google.com to view.

#8. 대회 규격으로 제출 파일 생성


대회 Overview에 공개된 객체별 행 형식을 세 모델에 똑같이 적용한다.

- 컬럼 순서: `annotation_id, image_id, category_id, bbox_x, bbox_y, bbox_w, bbox_h, score`
- 한 행은 검출 객체 하나를 뜻한다.
- `annotation_id`는 모델별 CSV 안에서 1부터 순서대로 부여한다.
- `image_id`는 Test 이미지 파일명의 숫자다.
- `category_id`는 해당 모델 매핑의 원본 category ID다.
- BBox는 원본 이미지 기준 `[x, y, width, height]`다.
- 이미지당 confidence 상위 예측 개수(Top-K)는 고정이 아닌, 각 모델별 validation 정책에서 설정한 값을 우선하여 적용한다.)

`smoke`에서는 세 모델의 형식과 좌표를 검증하지만 CSV를 제출용으로 확정하지 않는다. `full`에서만 YOLO11s / YOLO11m / YOLO12m 제출 CSV 3개를 저장한다.


In [14]:
show_stage_progress(13, "Build and validate three submission tables", "START")

def build_competition_submission(
    predictions_df,
    preprocessing_manifest,
    image_id_mapping,
):
    """공통 원시 예측을 대회가 요구하는 객체별 8개 컬럼으로 변환한다."""
    required_prediction_columns = {
        "file_name", "category_id", "score", "x", "y", "width", "height", "rank",
    }
    missing_prediction_columns = required_prediction_columns - set(predictions_df.columns)
    if missing_prediction_columns:
        raise ValueError(
            f"Raw prediction columns missing: {sorted(missing_prediction_columns)}"
        )

    dimensions = preprocessing_manifest[[
        "file_name", "source_width", "source_height",
    ]].copy()
    if dimensions["file_name"].duplicated().any():
        raise RuntimeError("Duplicate filenames were found in the preprocessing manifest.")

    if predictions_df.empty:
        empty_df = pd.DataFrame(columns=KAGGLE_SUBMISSION_COLUMNS)
        dtype_spec = {
            "annotation_id": np.int64,
            "image_id": np.int64,
            "category_id": np.int64,
            "bbox_x": float,
            "bbox_y": float,
            "bbox_w": float,
            "bbox_h": float,
            "score": float,
        }
        return empty_df.astype(dtype_spec)

    work = predictions_df.merge(
        image_id_mapping,
        on="file_name",
        how="left",
        validate="many_to_one",
    ).merge(
        dimensions,
        on="file_name",
        how="left",
        validate="many_to_one",
    )
    if work[["image_id", "source_width", "source_height"]].isna().any().any():
        unknown_files = work.loc[
            work["image_id"].isna() | work["source_width"].isna(),
            "file_name",
        ].unique().tolist()
        raise RuntimeError(f"Predictions could not be mapped to test images: {unknown_files[:10]}")

    work = work.sort_values(["image_id", "rank"], ascending=[True, True]).reset_index(drop=True)
    submission_df = pd.DataFrame({
        "annotation_id": np.arange(1, len(work) + 1, dtype=np.int64),
        "image_id": work["image_id"].astype(np.int64),
        "category_id": work["category_id"].astype(np.int64),
        "bbox_x": work["x"].astype(float),
        "bbox_y": work["y"].astype(float),
        "bbox_w": work["width"].astype(float),
        "bbox_h": work["height"].astype(float),
        "score": work["score"].astype(float),
    })
    return submission_df[KAGGLE_SUBMISSION_COLUMNS]


def validate_competition_submission(
    submission_df,
    preprocessing_manifest,
    image_id_mapping,
    allowed_category_ids,
    require_all_test_images,
):
    """공식 컬럼·ID·원본 좌표·confidence 계약을 검사한다. Top-K는 모델별 Validation 정책으로 이미 적용된다."""
    exact_columns = submission_df.columns.tolist() == list(KAGGLE_SUBMISSION_COLUMNS)
    required_test_count = EXPECTED_KAGGLE_TEST_IMAGES if require_all_test_images else len(
        preprocessing_manifest
    )
    active_mapping = image_id_mapping[
        image_id_mapping["file_name"].isin(preprocessing_manifest["file_name"])
    ].copy()
    active_mapping["image_id"] = active_mapping["image_id"].astype(np.int64)

    dimension_table = active_mapping.merge(
        preprocessing_manifest[["file_name", "source_width", "source_height"]],
        on="file_name",
        how="inner",
        validate="one_to_one",
    )
    dimension_by_image_id = dimension_table.set_index("image_id")[[
        "source_width", "source_height",
    ]]

    no_missing_values = not submission_df.isna().any().any()
    finite_numeric_values = True
    if not submission_df.empty:
        numeric_columns = KAGGLE_SUBMISSION_COLUMNS
        numeric_values = submission_df[numeric_columns].to_numpy(dtype=float)
        finite_numeric_values = bool(np.isfinite(numeric_values).all())

    expected_annotation_ids = list(range(1, len(submission_df) + 1))
    annotation_ids_valid = (
        submission_df["annotation_id"].astype(int).tolist() == expected_annotation_ids
        if exact_columns else False
    )
    allowed_image_ids = set(active_mapping["image_id"].astype(np.int64))
    submitted_image_ids = (
        set(submission_df["image_id"].astype(np.int64)) if exact_columns else set()
    )
    image_ids_valid = submitted_image_ids <= allowed_image_ids
    submitted_category_ids = (
        set(submission_df["category_id"].astype(np.int64)) if exact_columns else set()
    )
    category_ids_valid = submitted_category_ids <= set(map(int, allowed_category_ids))

    scores_valid = True
    boxes_positive = True
    boxes_inside_original_images = True
    maximum_objects = 0
    if exact_columns and not submission_df.empty:
        scores_valid = bool(submission_df["score"].between(0.0, 1.0, inclusive="both").all())
        boxes_positive = bool(
            (submission_df["bbox_x"] >= 0).all()
            and (submission_df["bbox_y"] >= 0).all()
            and (submission_df["bbox_w"] > 0).all()
            and (submission_df["bbox_h"] > 0).all()
        )
        checked = submission_df.merge(
            dimension_by_image_id,
            left_on="image_id",
            right_index=True,
            how="left",
            validate="many_to_one",
        )
        tolerance = 1e-4
        boxes_inside_original_images = bool(
            checked[["source_width", "source_height"]].notna().all().all()
            and (checked["bbox_x"] + checked["bbox_w"] <= checked["source_width"] + tolerance).all()
            and (checked["bbox_y"] + checked["bbox_h"] <= checked["source_height"] + tolerance).all()
        )
        maximum_objects = int(submission_df.groupby("image_id").size().max())

    # [수정] 이미지당 최대 검출 객출 수가 MAX_PILLS_PER_IMAGE 제한을 넘지 않는지 동적 검증
    per_image_count_valid = (
        MAX_PILLS_PER_IMAGE is None or maximum_objects <= MAX_PILLS_PER_IMAGE
    )

    checks = [
        {
            "check": "Exact competition column order",
            "status": "PASS" if exact_columns else "FAIL",
            "evidence": submission_df.columns.tolist(),
        },
        {
            "check": "Expected test image ID mapping count",
            "status": "PASS" if len(active_mapping) == required_test_count else "FAIL",
            "evidence": len(active_mapping),
        },
        {
            "check": "Unique numeric image IDs",
            "status": "PASS" if active_mapping["image_id"].is_unique else "FAIL",
            "evidence": active_mapping["image_id"].nunique(),
        },
        {
            "check": "At least one detected object",
            "status": "PASS" if len(submission_df) > 0 else "FAIL",
            "evidence": len(submission_df),
        },
        {
            "check": "No missing or non-finite values",
            "status": "PASS" if no_missing_values and finite_numeric_values else "FAIL",
            "evidence": len(submission_df),
        },
        {
            "check": "Sequential unique annotation IDs",
            "status": "PASS" if annotation_ids_valid else "FAIL",
            "evidence": len(expected_annotation_ids),
        },
        {
            "check": "Known test image IDs only",
            "status": "PASS" if image_ids_valid else "FAIL",
            "evidence": len(submitted_image_ids),
        },
        {
            "check": "Allowed original category IDs only",
            "status": "PASS" if category_ids_valid else "FAIL",
            "evidence": len(submitted_category_ids),
        },
        {
            "check": "Scores are within zero and one",
            "status": "PASS" if scores_valid else "FAIL",
            "evidence": float(submission_df["score"].min()) if len(submission_df) else None,
        },
        {
            "check": "Positive BBoxes inside original images",
            "status": "PASS" if boxes_positive and boxes_inside_original_images else "FAIL",
            "evidence": "original-image xywh",
        },
        {
            "check": "Per-image prediction count recorded",
            # [수정] 하드코딩된 "PASS" 대신 실제 검증 결과를 동적으로 할당
            "status": "PASS" if per_image_count_valid else "FAIL",
            "evidence": maximum_objects,
        },
    ]
    return pd.DataFrame(checks)


SUBMISSION_TABLES_BY_MODEL = {}
VALIDATION_TABLES_BY_MODEL = {}
validation_summary_frames = []

for model_name in tqdm(MODELS_TO_SUBMIT, desc="Validate model submissions", unit="model"):
    output = MODEL_INFERENCE_OUTPUTS[model_name]
    submission_df = build_competition_submission(
        output["raw_predictions_df"],
        preprocessing_manifest_df,
        test_image_id_mapping_df,
    )
    validation_table = validate_competition_submission(
        submission_df,
        preprocessing_manifest_df,
        test_image_id_mapping_df,
        output["allowed_original_category_ids"],
        require_all_test_images=(RUN_MODE == "full"),
    )
    SUBMISSION_TABLES_BY_MODEL[model_name] = submission_df
    VALIDATION_TABLES_BY_MODEL[model_name] = validation_table
    validation_summary_frames.append(validation_table.assign(model=model_name))

    print(f"{model_name} submission preview:")
    display(submission_df.head(10))
    display(validation_table)

ALL_MODEL_VALIDATION_TABLE = pd.concat(
    validation_summary_frames,
    ignore_index=True,
)[["model", "check", "status", "evidence"]]
print(f"Competition metric: {COMPETITION_METRIC}")
display(ALL_MODEL_VALIDATION_TABLE)
show_stage_progress(13, "Build and validate three submission tables", "DONE")

[Pipeline 13/14 |  93%] START: Build and validate three submission tables


Validate model submissions:   0%|          | 0/3 [00:00<?, ?model/s]

YOLO11s submission preview:


,annotation_id,image_id,category_id,bbox_x,bbox_y,bbox_w,bbox_h,score
0,1,1,1900,157.753866,250.408895,203.410238,126.043660,0.994859
1,2,1,27926,600.381022,670.140055,253.142904,484.254720,0.976642
2,3,1,24850,170.774902,739.183757,182.328044,292.986979,0.973402
3,4,1,16551,555.509766,71.107381,401.116862,404.509033,0.954651
4,5,3,1900,140.305237,242.160563,200.100932,129.607340,0.995252
5,6,3,27926,571.825195,626.509440,256.910807,489.792318,0.979458
6,7,3,24850,139.662537,702.413411,183.124003,293.490234,0.964288
7,8,3,16551,528.475179,63.590698,389.489583,396.588094,0.942314
8,9,4,1900,683.817871,807.809570,131.248210,207.857422,0.993273
9,10,4,24850,600.487386,251.322021,289.263997,164.616699,0.964814


,check,status,evidence
0,Exact competition column order,PASS,"[annotation_id, image_id, category_id, bbox_x,..."
1,Expected test image ID mapping count,PASS,842
2,Unique numeric image IDs,PASS,842
3,At least one detected object,PASS,4388
4,No missing or non-finite values,PASS,4388
5,Sequential unique annotation IDs,PASS,4388
6,Known test image IDs only,PASS,842
7,Allowed original category IDs only,PASS,88
8,Scores are within zero and one,PASS,0.001001
9,Positive BBoxes inside original images,PASS,original-image xywh


...

,annotation_id,image_id,category_id,bbox_x,bbox_y,bbox_w,bbox_h,score
0,1,1,27926,600.148926,670.292887,252.252604,485.555339,0.999942
1,2,1,1900,158.346842,251.465251,200.856527,125.251465,0.999644
2,3,1,24850,173.072998,740.480794,180.565674,292.034831,0.963829
3,4,1,16551,554.290527,72.249105,402.291016,404.756836,0.939428
4,5,1,16551,554.724935,76.566050,400.059245,197.740997,0.007666
5,6,1,16551,347.382324,44.987142,300.677572,435.122152,0.001753
6,7,1,1900,113.370443,248.169006,247.308594,359.580424,0.001659
7,8,1,1900,154.183757,247.964600,279.620280,322.389404,0.001323
8,9,1,1900,154.481934,251.990072,456.863770,195.216146,0.001146
9,10,3,27926,570.456055,625.730591,258.916016,492.606160,0.999985


,check,status,evidence
0,Exact competition column order,PASS,"[annotation_id, image_id, category_id, bbox_x,..."
1,Expected test image ID mapping count,PASS,842
2,Unique numeric image IDs,PASS,842
3,At least one detected object,PASS,4923
4,No missing or non-finite values,PASS,4923
5,Sequential unique annotation IDs,PASS,4923
6,Known test image IDs only,PASS,842
7,Allowed original category IDs only,PASS,91
8,Scores are within zero and one,PASS,0.001
9,Positive BBoxes inside original images,PASS,original-image xywh


Competition metric: mAP@[0.75:0.95]


,model,check,status,evidence
0,YOLO11s,Exact competition column order,PASS,"[annotation_id, image_id, category_id, bbox_x,..."
1,YOLO11s,Expected test image ID mapping count,PASS,842
2,YOLO11s,Unique numeric image IDs,PASS,842
3,YOLO11s,At least one detected object,PASS,4388
4,YOLO11s,No missing or non-finite values,PASS,4388
5,YOLO11s,Sequential unique annotation IDs,PASS,4388
6,YOLO11s,Known test image IDs only,PASS,842
7,YOLO11s,Allowed original category IDs only,PASS,88
8,YOLO11s,Scores are within zero and one,PASS,0.001001
9,YOLO11s,Positive BBoxes inside original images,PASS,original-image xywh


[Pipeline 13/14 |  93%] DONE: Build and validate three submission tables


In [15]:
show_stage_progress(14, "Export three CSV files and sync final artifacts", "START")

failed_validation = ALL_MODEL_VALIDATION_TABLE[
    ALL_MODEL_VALIDATION_TABLE["status"] != "PASS"
]
if not failed_validation.empty:
    failed_pairs = failed_validation[["model", "check"]].to_dict("records")
    raise RuntimeError(f"Submission validation failed: {failed_pairs}")

VALIDATION_SUMMARY_PATH = REPORT_DIR / f"submission_checks_{submission_batch_id}.csv"
ALL_MODEL_VALIDATION_TABLE.to_csv(
    VALIDATION_SUMMARY_PATH,
    index=False,
    encoding="utf-8-sig",
)

SUBMISSION_PATHS_BY_MODEL = {}
VALIDATION_REPORT_PATHS_BY_MODEL = {}

if RUN_MODE == "smoke":
    print("Three-model smoke inference completed successfully.")
    print("Set RUN_MODE='full', restart the runtime, and run all cells to create three CSV files.")
else:
    if len(preprocessing_manifest_df) != EXPECTED_KAGGLE_TEST_IMAGES:
        raise RuntimeError("Full submission requires all Kaggle test images.")

    for model_name in tqdm(MODELS_TO_SUBMIT, desc="Write Kaggle CSV files", unit="model"):
        output = MODEL_INFERENCE_OUTPUTS[model_name]
        submission_df = SUBMISSION_TABLES_BY_MODEL[model_name]
        inference_manifest_df = output["inference_manifest_df"]
        if inference_manifest_df["file_name"].nunique() != EXPECTED_KAGGLE_TEST_IMAGES:
            raise RuntimeError(f"{model_name} did not infer all test images.")

        submission_id = f"{model_name.lower()}_{submission_batch_id}"
        submission_path = SUBMISSION_DIR / f"submission_{submission_id}.csv"
        submission_df.to_csv(
            submission_path,
            index=False,
            encoding="utf-8",
            float_format="%.6f",
        )
        saved_submission_df = pd.read_csv(submission_path)
        if saved_submission_df.columns.tolist() != KAGGLE_SUBMISSION_COLUMNS:
            raise RuntimeError(f"{model_name} saved submission columns changed unexpectedly.")
        if len(saved_submission_df) != len(submission_df):
            raise RuntimeError(f"{model_name} saved submission row count changed unexpectedly.")

        predicted_image_count = int(submission_df["image_id"].nunique())
        validation_report = {
            "pipeline_version": PIPELINE_VERSION,
            "created_at": datetime.now().isoformat(),
            "competition_metric": COMPETITION_METRIC,
            "model_contract_path": FINAL_CONFIG_PATH,
            "model_name": model_name,
            "experiment_id": MODEL_RUN_CONFIGS[model_name]["experiment_id"],
            "checkpoint_sha256": MODEL_RUN_CONFIGS[model_name]["checkpoint_sha256"],
            "selected_confidence": MODEL_RUN_CONFIGS[model_name]["selected_confidence"],
            "selected_top_k": MODEL_RUN_CONFIGS[model_name].get("selected_top_k"),
            "use_tta": USE_TTA,
            "preprocessing_signature": preprocessing_signature,
            "raw_predictions_path": output["raw_predictions_path"],
            "image_id_mapping_path": TEST_IMAGE_ID_MAPPING_PATH,
            "submission_path": submission_path,
            "submission_sha256": sha256_file(submission_path),
            "submission_columns": KAGGLE_SUBMISSION_COLUMNS,
            "coordinate_format": "original-image xywh",
            "test_images": EXPECTED_KAGGLE_TEST_IMAGES,
            "images_with_predictions": predicted_image_count,
            "images_without_predictions": EXPECTED_KAGGLE_TEST_IMAGES - predicted_image_count,
            "detected_objects": len(submission_df),
            "checks": VALIDATION_TABLES_BY_MODEL[model_name].to_dict("records"),
        }
        validation_report_path = REPORT_DIR / f"submission_validation_{submission_id}.json"
        write_json_atomic(validation_report_path, validation_report)
        SUBMISSION_PATHS_BY_MODEL[model_name] = submission_path
        VALIDATION_REPORT_PATHS_BY_MODEL[model_name] = validation_report_path
        print(f"{model_name} submission: {submission_path}")


# Kaggle 첫 제출용으로 전체 1등 단일 모델 CSV를 명확한 이름으로 하나 더 만든다.
RECOMMENDED_SUBMISSION_PATH = None
if RUN_MODE == "full":
    recommended_source = Path(SUBMISSION_PATHS_BY_MODEL[RECOMMENDED_MODEL])
    RECOMMENDED_SUBMISSION_PATH = (
        SUBMISSION_DIR
        / f"submission_RECOMMENDED_{RECOMMENDED_MODEL.lower()}_{submission_batch_id}.csv"
    )
    shutil.copy2(recommended_source, RECOMMENDED_SUBMISSION_PATH)
    if sha256_file(recommended_source) != sha256_file(RECOMMENDED_SUBMISSION_PATH):
        raise RuntimeError("Recommended submission copy SHA256 mismatch.")
    print(
        f"RECOMMENDED FIRST KAGGLE SUBMISSION: {RECOMMENDED_MODEL} -> "
        f"{RECOMMENDED_SUBMISSION_PATH}"
    )


pipeline_seconds = time.perf_counter() - pipeline_timer_started_perf
pipeline_finished_at = datetime.now()

KAGGLE_TIMING_JSON_PATH = REPORT_DIR / f"kaggle_timing_{submission_batch_id}.json"
KAGGLE_TIMING_CSV_PATH = REPORT_DIR / f"kaggle_timing_{submission_batch_id}.csv"
kaggle_timing_table = model_inference_timing_table.copy()
kaggle_timing_table["run_mode"] = RUN_MODE
kaggle_timing_table["preprocessing_seconds_shared"] = preprocessing_seconds
kaggle_timing_table["pipeline_seconds_total"] = pipeline_seconds
kaggle_timing_table.to_csv(
    KAGGLE_TIMING_CSV_PATH,
    index=False,
    encoding="utf-8-sig",
)
kaggle_timing_record = {
    "schema_version": 2,
    "pipeline_version": PIPELINE_VERSION,
    "run_mode": RUN_MODE,
    "models": list(MODELS_TO_SUBMIT),
    "recommended_model": RECOMMENDED_MODEL,
    "recommended_competition_mAP": RECOMMENDED_COMPETITION_MAP,
    "images_per_model": len(active_test_image_paths),
    "storage_mode": "colab_local_processing_drive_final_outputs",
    "input_staging_mode": input_staging_mode,
    "preprocessing_cache_reused": bool(reuse_preprocessing),
    "pipeline_started_at": pipeline_timer_started_at.isoformat(),
    "pipeline_finished_at": pipeline_finished_at.isoformat(),
    "preprocessing_seconds": preprocessing_seconds,
    "pipeline_seconds": pipeline_seconds,
    "model_inference": model_inference_timing_table.to_dict("records"),
    "submission_paths": SUBMISSION_PATHS_BY_MODEL,
}
write_json_atomic(KAGGLE_TIMING_JSON_PATH, kaggle_timing_record)
display(kaggle_timing_table)

# 전처리 이미지는 로컬에만 두고 재현·제출에 필요한 작은 산출물만 Drive에 보존한다.
artifact_targets = [
    (FINAL_CONFIG_PATH, DRIVE_FINAL_CONFIG_DIR),
    (LOCAL_STAGING_MANIFEST_PATH, DRIVE_PREPROCESSING_MANIFEST_DIR),
    (TEST_IMAGE_ID_MAPPING_PATH, DRIVE_PREPROCESSING_MANIFEST_DIR),
    (PREPROCESSING_MANIFEST_PATH, DRIVE_PREPROCESSING_MANIFEST_DIR),
    (VALIDATION_SUMMARY_PATH, DRIVE_REPORT_DIR),
    (KAGGLE_TIMING_JSON_PATH, DRIVE_REPORT_DIR),
    (KAGGLE_TIMING_CSV_PATH, DRIVE_REPORT_DIR),
]
if RUN_MODE == "full" and RECOMMENDED_SUBMISSION_PATH is not None:
    artifact_targets.append(
        (RECOMMENDED_SUBMISSION_PATH, DRIVE_SUBMISSION_DIR)
    )

for model_name in MODELS_TO_SUBMIT:
    output = MODEL_INFERENCE_OUTPUTS[model_name]
    artifact_targets.extend([
        (output["raw_predictions_path"], DRIVE_PREDICTION_DIR),
        (output["inference_manifest_path"], DRIVE_PREDICTION_DIR),
        (output["cache_marker_path"], DRIVE_PREDICTION_DIR),
        (PREVIEW_PATHS_BY_MODEL[model_name], DRIVE_FIGURE_DIR),
    ])
    if RUN_MODE == "full":
        artifact_targets.extend([
            (SUBMISSION_PATHS_BY_MODEL[model_name], DRIVE_SUBMISSION_DIR),
            (VALIDATION_REPORT_PATHS_BY_MODEL[model_name], DRIVE_REPORT_DIR),
        ])

drive_artifact_rows = []
for local_artifact_path, drive_directory in tqdm(
    artifact_targets,
    desc="Copy final artifacts to Drive",
    unit="file",
):
    drive_artifact_path = persist_artifact_to_drive(local_artifact_path, drive_directory)
    drive_artifact_rows.append({
        "local_path": str(local_artifact_path),
        "drive_path": str(drive_artifact_path),
        "sha256": sha256_file(local_artifact_path),
        "size_bytes": Path(local_artifact_path).stat().st_size,
    })

DRIVE_SYNC_MANIFEST_PATH = REPORT_DIR / f"drive_sync_{submission_batch_id}.json"
write_json_atomic(DRIVE_SYNC_MANIFEST_PATH, {
    "created_at": datetime.now().isoformat(),
    "storage_mode": "colab_local_processing_drive_final_outputs",
    "preprocessed_images_copied_to_drive": False,
    "models": list(MODELS_TO_SUBMIT),
    "artifacts": drive_artifact_rows,
})
drive_sync_path = persist_artifact_to_drive(DRIVE_SYNC_MANIFEST_PATH, DRIVE_REPORT_DIR)

print("Final artifacts were copied to Google Drive.")
print(f"Drive sync manifest: {drive_sync_path}")
if RUN_MODE == "full":
    recommended_drive_path = next(
        row["drive_path"] for row in drive_artifact_rows
        if RECOMMENDED_SUBMISSION_PATH is not None
        and Path(row["local_path"]) == Path(RECOMMENDED_SUBMISSION_PATH)
    )
    print("=" * 80)
    print(
        f"FIRST KAGGLE SUBMISSION RECOMMENDATION: {RECOMMENDED_MODEL} "
        f"({COMPETITION_METRIC}={RECOMMENDED_COMPETITION_MAP:.6f})"
    )
    print(f"Recommended CSV: {recommended_drive_path}")
    print("Other architecture-winner CSV files (optional additional submissions):")
    for model_name in MODELS_TO_SUBMIT:
        local_submission = Path(SUBMISSION_PATHS_BY_MODEL[model_name])
        drive_submission = next(
            row["drive_path"] for row in drive_artifact_rows
            if Path(row["local_path"]) == local_submission
        )
        print(f"- {model_name}: {drive_submission}")

show_stage_progress(14, "Export three CSV files and sync final artifacts", "DONE")

[Pipeline 14/14 | 100%] START: Export three CSV files and sync final artifacts


Write Kaggle CSV files:   0%|          | 0/3 [00:00<?, ?model/s]

YOLO11s submission: /content/baby_kangaroo_kaggle_submission_v21/results/submissions/submission_yolo11s_20260819_022523_8380680e9a2ed00c_c19d2288912a96f9.csv
YOLO11m submission: /content/baby_kangaroo_kaggle_submission_v21/results/submissions/submission_yolo11m_20260819_022523_8380680e9a2ed00c_c19d2288912a96f9.csv
YOLO12m submission: /content/baby_kangaroo_kaggle_submission_v21/results/submissions/submission_yolo12m_20260819_022523_8380680e9a2ed00c_c19d2288912a96f9.csv
RECOMMENDED FIRST KAGGLE SUBMISSION: YOLO11m -> /content/baby_kangaroo_kaggle_submission_v21/results/submissions/submission_RECOMMENDED_yolo11m_20260819_022523_8380680e9a2ed00c_c19d2288912a96f9.csv


,model,images,detected_objects,selected_confidence,selected_top_k,inference_seconds,inference_minutes,peak_gpu_memory_gb,cache_reused,run_mode,preprocessing_seconds_shared,pipeline_seconds_total
0,YOLO11s,842,4388,0.001,6,33.739083,0.562318,1.075649,False,full,253.118529,423.98264
1,YOLO11m,842,4228,0.001,6,34.884859,0.581414,1.856948,False,full,253.118529,423.98264
2,YOLO12m,842,4923,0.001,12,38.116497,0.635275,2.523273,False,full,253.118529,423.98264


Copy final artifacts to Drive:   0%|          | 0/26 [00:00<?, ?file/s]

Copied to Colab local storage: v605_submission_config_8380680e9a2ed00c.json
Reused verified local file: /content/drive/MyDrive/baby_kangaroo/week2/kaggle_submission_v2/preprocessing_manifests/local_input_staging_manifest.csv
Reused verified local file: /content/drive/MyDrive/baby_kangaroo/week2/kaggle_submission_v2/preprocessing_manifests/test_image_id_mapping.csv
Copied to Colab local storage: preprocessing_manifest.csv
Copied to Colab local storage: submission_checks_20260819_022523_8380680e9a2ed00c_c19d2288912a96f9.csv
Copied to Colab local storage: kaggle_timing_20260819_022523_8380680e9a2ed00c_c19d2288912a96f9.json
Copied to Colab local storage: kaggle_timing_20260819_022523_8380680e9a2ed00c_c19d2288912a96f9.csv
Copied to Colab local storage: submission_RECOMMENDED_yolo11m_20260819_022523_8380680e9a2ed00c_c19d2288912a96f9.csv
Copied to Colab local storage: raw_predictions_yolo11s_290159bd2bebd828.csv
Copied to Colab local storage: inference_manifest_yolo11s_290159bd2bebd828.csv
Co

## Kaggle에 올리기 전 마지막 확인

1. 먼저 `RUN_MODE="smoke"`로 8장 실행한다.
2. 세 evaluation summary가 모두 읽히고 `MODEL_SELECTION_TABLE`이 표시되는지 확인한다.
3. `class_mapping.csv`가 정확히 118개, YOLO ID 0..117인지 확인한다.
4. Test 이미지 수가 842장인지 확인한다.
5. 예측 미리보기에서 BBox가 실제 알약 위치에 맞는지 확인한다.
6. 모든 submission validation 항목이 PASS인지 확인한다.
7. 이상이 없으면 `RUN_MODE="full"`로 런타임을 새로 시작해 전체 실행한다.

최종 CSV 컬럼 순서는 정확히 아래와 같아야 한다.

```text
annotation_id,image_id,category_id,bbox_x,bbox_y,bbox_w,bbox_h,score
```

Drive 저장 위치:

```text
/content/drive/MyDrive/baby_kangaroo/week2/kaggle_submission_v2/submissions/
```

세 architecture winner CSV 외에 전체 competition 평가 1등 모델의 파일이

```text
submission_RECOMMENDED_<model>_....csv
```

이름으로 한 번 더 생성된다.

**첫 Kaggle baseline 제출은 `submission_RECOMMENDED_...csv`를 사용한다.**
WBF/TTA는 이 baseline 제출이 정상임을 확인한 뒤 별도 실험으로 진행한다.
